In [ ]:
# 导入必要的库
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import os
import rasterio
from rasterio.transform import from_origin
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("开始加载数据...")

# 1. 加载数据 - 尝试不同的编码格式
def load_csv_with_encoding(file_path):
    """
    尝试多种编码格式加载CSV文件
    """
    encodings = ['utf-8', 'gbk', 'gb2312', 'latin-1', 'cp1252']
    
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"成功使用 {encoding} 编码加载文件: {file_path}")
            return df
        except UnicodeDecodeError:
            print(f"编码 {encoding} 失败，尝试下一个...")
            continue
        except Exception as e:
            print(f"编码 {encoding} 出现其他错误: {str(e)}")
            continue
    
    raise Exception(f"无法使用任何编码格式加载文件: {file_path}")

# 加载数据
try:
    bus_data = load_csv_with_encoding(r'E:\chuli\traffic\HK_busStation.csv')
    school_data = load_csv_with_encoding(r'E:\chuli\over\school_HK.csv')
except Exception as e:
    print(f"加载数据时出错: {str(e)}")
    print("请检查文件路径和文件格式")
    raise

print(f"公交站数据形状: {bus_data.shape}")
print(f"学校数据形状: {school_data.shape}")

# 检查列名
print("公交站数据列名:", bus_data.columns.tolist())
print("学校数据列名:", school_data.columns.tolist())

# 2. 数据预处理 - 假设经纬度列名
# 根据常见列名模式进行处理
def find_coord_columns(df, data_type):
    """自动查找坐标列"""
    lon_patterns = ['longitude', 'lon', 'lng', 'x', 'X', 'LONG', 'LON', '经度', 'x坐标']
    lat_patterns = ['latitude', 'lat', 'y', 'Y', 'LAT', '纬度', 'y坐标']
    
    lon_col = None
    lat_col = None
    
    for pattern in lon_patterns:
        for col in df.columns:
            if pattern.lower() in col.lower():
                lon_col = col
                break
        if lon_col:
            break
    
    for pattern in lat_patterns:
        for col in df.columns:
            if pattern.lower() in col.lower():
                lat_col = col
                break
        if lat_col:
            break
    
    if lon_col is None or lat_col is None:
        # 如果没找到标准列名，使用前两列
        cols = df.columns.tolist()
        if len(cols) >= 2:
            lon_col, lat_col = cols[0], cols[1]
            print(f"警告: {data_type}数据未找到标准坐标列名，使用前两列: {lon_col}, {lat_col}")
        else:
            raise ValueError(f"{data_type}数据列数不足，至少需要2列坐标数据")
    
    return lon_col, lat_col

try:
    bus_lon_col, bus_lat_col = find_coord_columns(bus_data, "公交站")
    school_lon_col, school_lat_col = find_coord_columns(school_data, "学校")
except ValueError as e:
    print(f"数据列名错误: {str(e)}")
    print("请确保CSV文件包含经纬度坐标列")
    raise

print(f"公交站经度列: {bus_lon_col}, 纬度列: {bus_lat_col}")
print(f"学校经度列: {school_lon_col}, 纬度列: {school_lat_col}")

# 3. 数据清洗和验证
def clean_coordinate_data(df, lon_col, lat_col, data_type):
    """
    清洗坐标数据，移除无效值
    """
    print(f"清洗{data_type}坐标数据...")
    
    # 检查并转换坐标数据类型
    try:
        df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
        df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    except Exception as e:
        print(f"转换{data_type}坐标数据类型时出错: {str(e)}")
        raise
    
    # 移除无效坐标（NaN, 无穷大等）
    original_count = len(df)
    df = df.dropna(subset=[lon_col, lat_col])
    
    # 检查坐标范围（香港的经纬度范围）
    df = df[(df[lon_col] >= 113.8) & (df[lon_col] <= 114.4) & 
            (df[lat_col] >= 22.1) & (df[lat_col] <= 22.6)]
    
    removed_count = original_count - len(df)
    print(f"{data_type}数据清洗完成: 原始{original_count}条，移除{removed_count}条无效数据，剩余{len(df)}条")
    
    return df

# 清洗数据
bus_data = clean_coordinate_data(bus_data, bus_lon_col, bus_lat_col, "公交站")
school_data = clean_coordinate_data(school_data, school_lon_col, school_lat_col, "学校")

# 4. 创建GeoDataFrame
bus_geometry = [Point(xy) for xy in zip(bus_data[bus_lon_col], bus_data[bus_lat_col])]
bus_gdf = gpd.GeoDataFrame(bus_data, geometry=bus_geometry, crs='EPSG:4326')

school_geometry = [Point(xy) for xy in zip(school_data[school_lon_col], school_data[school_lat_col])]
school_gdf = gpd.GeoDataFrame(school_data, geometry=school_geometry, crs='EPSG:4326')

print("数据加载和清洗完成")

# 5. 计算学校到最近公交站的距离
def haversine_distance(lon1, lat1, lon2, lat2):
    """
    使用Haversine公式计算两点间的距离（米）
    """
    # 转换为弧度
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    
    # Haversine公式
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    # 地球半径（米）
    r = 6371000
    
    return c * r

def calculate_accessibility(bus_gdf, school_gdf):
    """
    计算学校到最近公交站的距离
    使用KDTree进行快速最近邻搜索
    """
    print("开始计算学校到最近公交站的距离...")
    
    if len(bus_gdf) == 0 or len(school_gdf) == 0:
        raise ValueError("公交站或学校数据为空，无法计算可达性")
    
    # 提取坐标
    bus_coords = np.array([[point.x, point.y] for point in bus_gdf.geometry])
    school_coords = np.array([[point.x, point.y] for point in school_gdf.geometry])
    
    # 使用KDTree进行最近邻搜索
    bus_tree = cKDTree(bus_coords)
    
    # 计算每个学校到最近公交站的距离
    distances_deg, indices = bus_tree.query(school_coords)
    
    # 使用Haversine公式计算精确距离
    distances_meters = []
    for i, idx in enumerate(indices):
        dist_m = haversine_distance(
            school_coords[i][0], school_coords[i][1],
            bus_coords[idx][0], bus_coords[idx][1]
        )
        distances_meters.append(dist_m)
    
    distances_meters = np.array(distances_meters)
    
    # 添加距离信息到学校GeoDataFrame
    school_gdf['nearest_bus_distance_m'] = distances_meters
    school_gdf['nearest_bus_index'] = indices
    
    # 获取最近公交站的信息
    nearest_bus_names = bus_gdf.iloc[indices][bus_gdf.columns[0]].values
    school_gdf['nearest_bus_station'] = nearest_bus_names
    
    print(f"距离计算完成，最小距离: {distances_meters.min():.2f}米, 最大距离: {distances_meters.max():.2f}米")
    print(f"平均距离: {distances_meters.mean():.2f}米")
    
    return school_gdf, bus_gdf

# 执行距离计算
try:
    school_with_accessibility, bus_gdf = calculate_accessibility(bus_gdf, school_gdf)
except ValueError as e:
    print(f"计算可达性时出错: {str(e)}")
    raise

# 6. 创建栅格化函数
def create_accessibility_grid(school_gdf, bus_gdf, cell_size=500, buffer_distance=5000):
    """
    将学校可达性数据转换为栅格
    使用反距离加权插值方法
    """
    print("开始创建栅格数据...")
    
    if len(school_gdf) == 0 or len(bus_gdf) == 0:
        raise ValueError("学校或公交站数据为空，无法创建栅格")
    
    # 获取研究区域边界
    school_bounds = school_gdf.total_bounds  # (minx, miny, maxx, maxy)
    bus_bounds = bus_gdf.total_bounds
    
    # 综合考虑学校和公交站的边界
    minx = min(school_bounds[0], bus_bounds[0])
    miny = min(school_bounds[1], bus_bounds[1])
    maxx = max(school_bounds[2], bus_bounds[2])
    maxy = max(school_bounds[3], bus_bounds[3])
    
    # 扩展边界以包含缓冲区
    lat_factor = np.cos(np.radians((miny + maxy) / 2))
    minx -= buffer_distance / (111000 * lat_factor)
    miny -= buffer_distance / 111000
    maxx += buffer_distance / (111000 * lat_factor)
    maxy += buffer_distance / 111000
    
    # 计算栅格尺寸（使用Haversine距离转换）
    x_range = haversine_distance(minx, (miny + maxy) / 2, maxx, (miny + maxy) / 2)
    y_range = haversine_distance((minx + maxx) / 2, miny, (minx + maxx) / 2, maxy)
    
    width = max(10, int(x_range / cell_size))  # 确保至少有10个像素
    height = max(10, int(y_range / cell_size))
    
    print(f"研究区域范围: ({minx:.6f}, {miny:.6f}, {maxx:.6f}, {maxy:.6f})")
    print(f"栅格尺寸: {width} x {height}, 像素大小: {cell_size}米")
    
    # 创建坐标网格
    x_coords = np.linspace(minx, maxx, width)
    y_coords = np.linspace(maxy, miny, height)  # 注意：y轴从上到下
    xx, yy = np.meshgrid(x_coords, y_coords)
    
    # 展平坐标用于距离计算
    grid_points = np.column_stack([xx.ravel(), yy.ravel()])
    
    # 提取学校坐标和距离值
    school_coords = np.array([[point.x, point.y] for point in school_gdf.geometry])
    school_distances = school_gdf['nearest_bus_distance_m'].values
    
    # 创建KDTree
    school_tree = cKDTree(school_coords)
    
    # 对于每个栅格点，计算插值
    accessibility_grid = np.full(len(grid_points), np.nan)
    
    # 设置插值参数
    k_neighbors = min(10, len(school_coords))  # 使用最近的10个点
    power = 2  # IDW的幂参数
    
    print("正在进行空间插值...")
    for i in range(len(grid_points)):
        if i % 10000 == 0:  # 每处理10000个点打印一次进度
            print(f"已处理 {i}/{len(grid_points)} 个栅格点")
        
        # 找到最近的k个学校点
        distances_to_schools, indices = school_tree.query(grid_points[i], k=k_neighbors)
        
        # 处理单个点的情况
        if np.isscalar(distances_to_schools):
            distances_to_schools = np.array([distances_to_schools])
            indices = np.array([indices])
        
        # 计算加权平均
        weights = []
        values = []
        
        for j, (dist, idx) in enumerate(zip(distances_to_schools, indices)):
            if dist < 1e-6:  # 如果距离非常小，直接使用该点的值
                accessibility_grid[i] = school_distances[idx]
                break
            else:
                # 使用反距离加权
                weight = 1 / (dist ** power)
                weights.append(weight)
                values.append(school_distances[idx])
        else:
            # 如果没有距离为0的点，则计算加权平均
            if len(weights) > 0:
                total_weight = sum(weights)
                if total_weight > 0:
                    accessibility_grid[i] = sum(w * v for w, v in zip(weights, values)) / total_weight
                else:
                    accessibility_grid[i] = np.nan
            else:
                accessibility_grid[i] = np.nan
    
    # 重塑为2D栅格
    accessibility_grid = accessibility_grid.reshape(height, width)
    
    # 计算地理变换参数
    x_pixel_size = (maxx - minx) / width
    y_pixel_size = (maxy - miny) / height  # 注意这里是负值，因为y轴从上到下
    transform = from_origin(minx, maxy, x_pixel_size, abs(y_pixel_size))
    
    print("栅格数据创建完成")
    return accessibility_grid, transform, (minx, miny, maxx, maxy)

# 创建栅格数据
try:
    accessibility_grid, transform, bounds = create_accessibility_grid(school_with_accessibility, bus_gdf, cell_size=10)
except Exception as e:
    print(f"创建栅格时出错: {str(e)}")
    raise

# 7. 输出栅格数据
def save_raster_data(grid, transform, bounds, output_path):
    """
    保存栅格数据为GeoTIFF格式
    """
    print(f"开始保存栅格数据到: {output_path}")
    
    # 确保输出目录存在
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    with rasterio.open(
        output_path,
        'w',
        driver='GTiff',
        height=grid.shape[0],
        width=grid.shape[1],
        count=1,
        dtype=rasterio.float32,
        crs='EPSG:4326',
        transform=transform,
        nodata=-9999
    ) as dst:
        # 将NaN值替换为nodata值
        grid_masked = np.where(np.isnan(grid), -9999, grid).astype(np.float32)
        dst.write(grid_masked, 1)
    
    print("栅格数据保存完成")

# 保存可达性栅格
output_raster_path = r'E:\chuli\accessibility\school_bus_accessibility.tif'
try:
    save_raster_data(accessibility_grid, transform, bounds, output_raster_path)
except Exception as e:
    print(f"保存栅格数据时出错: {str(e)}")
    raise

# 8. 可视化结果
def visualize_results(school_gdf, bus_gdf, accessibility_grid, bounds, save_path=None):
    """
    可视化可达性分析结果
    """
    print("开始可视化...")
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    # 左图：原始数据分布
    ax1 = axes[0]
    ax1.scatter(bus_gdf.geometry.x, bus_gdf.geometry.y, c='blue', s=1, alpha=0.6, label='公交站')
    ax1.scatter(school_gdf.geometry.x, school_gdf.geometry.y, c='red', s=20, alpha=0.8, label='学校')
    ax1.set_title('学校与公交站分布')
    ax1.set_xlabel('经度')
    ax1.set_ylabel('纬度')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 右图：可达性栅格
    ax2 = axes[1]
    # 过滤掉NaN值用于可视化
    valid_data = accessibility_grid[~np.isnan(accessibility_grid)]
    if len(valid_data) > 0:
        vmax = np.nanpercentile(accessibility_grid, 95)
        im = ax2.imshow(accessibility_grid, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], 
                        cmap='RdYlBu_r', origin='upper', vmin=0, vmax=vmax)
    else:
        im = ax2.imshow(accessibility_grid, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], 
                        cmap='RdYlBu_r', origin='upper')
    
    ax2.scatter(school_gdf.geometry.x, school_gdf.geometry.y, c='black', s=10, alpha=0.8, label='学校')
    ax2.set_title('学校到最近公交站距离（米）')
    ax2.set_xlabel('经度')
    ax2.set_ylabel('纬度')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.colorbar(im, ax=ax2, label='距离（米）', shrink=0.8)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"可视化结果已保存到: {save_path}")
    
    plt.show()

# 可视化结果
viz_path = r'E:\chuli\accessibility\accessibility_visualization.png'
os.makedirs(os.path.dirname(viz_path), exist_ok=True)
try:
    visualize_results(school_with_accessibility, bus_gdf, accessibility_grid, bounds, viz_path)
except Exception as e:
    print(f"可视化时出错: {str(e)}")

# 9. 生成统计报告
def generate_statistics(school_gdf):
    """
    生成可达性统计报告
    """
    distances = school_gdf['nearest_bus_distance_m'].dropna()
    
    if len(distances) == 0:
        print("警告: 没有有效的距离数据")
        return {}
    
    stats = {
        '学校总数': len(school_gdf),
        '有公交可达的学校数': len(distances),
        '平均距离（米）': distances.mean(),
        '中位数距离（米）': distances.median(),
        '最短距离（米）': distances.min(),
        '最长距离（米）': distances.max(),
        '标准差（米）': distances.std(),
        '距离小于500米的学校数': (distances < 500).sum(),
        '距离小于1000米的学校数': (distances < 1000).sum(),
        '距离小于2000米的学校数': (distances < 2000).sum(),
        '距离小于3000米的学校数': (distances < 3000).sum()
    }
    
    print("\n=== 可达性统计报告 ===")
    for key, value in stats.items():
        if isinstance(value, float):
            print(f"{key}: {value:.2f}")
        else:
            print(f"{key}: {value}")
    
    return stats

# 生成统计报告
stats = generate_statistics(school_with_accessibility)

# 10. 保存处理后的学校数据
school_output_path = r'E:\chuli\accessibility\school_with_accessibility.csv'
try:
    school_with_accessibility.to_csv(school_output_path, index=False, encoding='utf-8-sig')
    print(f"\n学校可达性数据已保存到: {school_output_path}")
except Exception as e:
    print(f"保存学校数据时出错: {str(e)}")

# 11. 创建不同距离阈值的二进制栅格
def create_binary_accessibility_grids(accessibility_grid, transform, bounds, thresholds=[500, 1000, 2000]):
    """
    创建不同距离阈值的二进制可达性栅格
    """
    print("创建不同距离阈值的二进制栅格...")
    
    for threshold in thresholds:
        # 创建二进制栅格（可达=1，不可达=0），处理NaN值
        binary_grid = np.where(np.isnan(accessibility_grid), 0, 
                              np.where(accessibility_grid <= threshold, 1, 0)).astype(np.int8)
        
        # 保存二进制栅格
        binary_path = f'E:\\chuli\\accessibility\\accessibility_within_{threshold}m.tif'
        try:
            save_raster_data(binary_grid, transform, bounds, binary_path)
            print(f"距离阈值{threshold}米的二进制栅格已保存: {binary_path}")
        except Exception as e:
            print(f"保存二进制栅格 {binary_path} 时出错: {str(e)}")

# 创建二进制栅格
create_binary_accessibility_grids(accessibility_grid, transform, bounds, [500, 1000, 2000])

print(f"\n=== 分析完成 ===")
print(f"栅格文件位置: {output_raster_path}")
print(f"可视化文件位置: {viz_path}")
print(f"学校数据文件位置: {school_output_path}")

# 12. 显示栅格基本信息
print(f"\n=== 栅格数据信息 ===")
print(f"栅格尺寸: {accessibility_grid.shape}")
valid_data = accessibility_grid[~np.isnan(accessibility_grid)]
if len(valid_data) > 0:
    print(f"有效数据范围: {np.nanmin(accessibility_grid):.2f} - {np.nanmax(accessibility_grid):.2f} 米")
print(f"NaN值数量: {np.count_nonzero(np.isnan(accessibility_grid))}")
print(f"有效值数量: {np.count_nonzero(~np.isnan(accessibility_grid))}")

print("\n所有处理完成！")


In [ ]:
# 导入必要的库
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import os
import rasterio
from rasterio.transform import from_origin
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("开始加载数据...")

# 1. 加载数据 - 尝试不同的编码格式
def load_csv_with_encoding(file_path):
    """
    尝试多种编码格式加载CSV文件
    """
    encodings = ['utf-8', 'gbk', 'gb2312', 'latin-1', 'cp1252']
    
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"成功使用 {encoding} 编码加载文件: {file_path}")
            return df
        except UnicodeDecodeError:
            print(f"编码 {encoding} 失败，尝试下一个...")
            continue
        except Exception as e:
            print(f"编码 {encoding} 出现其他错误: {str(e)}")
            continue
    
    raise Exception(f"无法使用任何编码格式加载文件: {file_path}")

# 加载数据
try:
    shop_data = load_csv_with_encoding(r'E:\chuli\traffic\HK_shop.csv')
    school_data = load_csv_with_encoding(r'E:\chuli\over\school_HK.csv')
except Exception as e:
    print(f"加载数据时出错: {str(e)}")
    print("请检查文件路径和文件格式")
    raise

print(f"商店数据形状: {shop_data.shape}")
print(f"学校数据形状: {school_data.shape}")

# 检查列名
print("商店数据列名:", shop_data.columns.tolist())
print("学校数据列名:", school_data.columns.tolist())

# 2. 数据预处理 - 假设经纬度列名
def find_coord_columns(df, data_type):
    """自动查找坐标列"""
    lon_patterns = ['longitude', 'lon', 'lng', 'x', 'X', 'LONG', 'LON', '经度', 'x坐标', 'X坐标']
    lat_patterns = ['latitude', 'lat', 'y', 'Y', 'LAT', '纬度', 'y坐标', 'Y坐标']
    
    lon_col = None
    lat_col = None
    
    for pattern in lon_patterns:
        for col in df.columns:
            if pattern.lower() in col.lower():
                lon_col = col
                break
        if lon_col:
            break
    
    for pattern in lat_patterns:
        for col in df.columns:
            if pattern.lower() in col.lower():
                lat_col = col
                break
        if lat_col:
            break
    
    if lon_col is None or lat_col is None:
        # 如果没找到标准列名，使用前两列
        cols = df.columns.tolist()
        if len(cols) >= 2:
            lon_col, lat_col = cols[0], cols[1]
            print(f"警告: {data_type}数据未找到标准坐标列名，使用前两列: {lon_col}, {lat_col}")
        else:
            raise ValueError(f"{data_type}数据列数不足，至少需要2列坐标数据")
    
    return lon_col, lat_col

try:
    shop_lon_col, shop_lat_col = find_coord_columns(shop_data, "商店")
    school_lon_col, school_lat_col = find_coord_columns(school_data, "学校")
except ValueError as e:
    print(f"数据列名错误: {str(e)}")
    print("请确保CSV文件包含经纬度坐标列")
    raise

print(f"商店经度列: {shop_lon_col}, 纬度列: {shop_lat_col}")
print(f"学校经度列: {school_lon_col}, 纬度列: {school_lat_col}")

# 3. 数据清洗和验证
def clean_coordinate_data(df, lon_col, lat_col, data_type):
    """
    清洗坐标数据，移除无效值
    """
    print(f"清洗{data_type}坐标数据...")
    
    # 检查并转换坐标数据类型
    try:
        df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
        df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    except Exception as e:
        print(f"转换{data_type}坐标数据类型时出错: {str(e)}")
        raise
    
    # 移除无效坐标（NaN, 无穷大等）
    original_count = len(df)
    df = df.dropna(subset=[lon_col, lat_col])
    
    # 检查坐标范围（香港的经纬度范围）
    df = df[(df[lon_col] >= 113.8) & (df[lon_col] <= 114.4) & 
            (df[lat_col] >= 22.1) & (df[lat_col] <= 22.6)]
    
    removed_count = original_count - len(df)
    print(f"{data_type}数据清洗完成: 原始{original_count}条，移除{removed_count}条无效数据，剩余{len(df)}条")
    
    return df

# 清洗数据
shop_data = clean_coordinate_data(shop_data, shop_lon_col, shop_lat_col, "商店")
school_data = clean_coordinate_data(school_data, school_lon_col, school_lat_col, "学校")

# 4. 创建GeoDataFrame
shop_geometry = [Point(xy) for xy in zip(shop_data[shop_lon_col], shop_data[shop_lat_col])]
shop_gdf = gpd.GeoDataFrame(shop_data, geometry=shop_geometry, crs='EPSG:4326')

school_geometry = [Point(xy) for xy in zip(school_data[school_lon_col], school_data[school_lat_col])]
school_gdf = gpd.GeoDataFrame(school_data, geometry=school_geometry, crs='EPSG:4326')

print("数据加载和清洗完成")

# 5. 计算学校到最近商店的距离
def haversine_distance(lon1, lat1, lon2, lat2):
    """
    使用Haversine公式计算两点间的距离（米）
    """
    # 转换为弧度
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    
    # Haversine公式
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    # 地球半径（米）
    r = 6371000
    
    return c * r

def calculate_shop_accessibility(shop_gdf, school_gdf):
    """
    计算学校到最近商店的距离
    使用KDTree进行快速最近邻搜索
    """
    print("开始计算学校到最近商店的距离...")
    
    if len(shop_gdf) == 0 or len(school_gdf) == 0:
        raise ValueError("商店或学校数据为空，无法计算可达性")
    
    # 提取坐标
    shop_coords = np.array([[point.x, point.y] for point in shop_gdf.geometry])
    school_coords = np.array([[point.x, point.y] for point in school_gdf.geometry])
    
    # 使用KDTree进行最近邻搜索
    shop_tree = cKDTree(shop_coords)
    
    # 计算每个学校到最近商店的距离
    distances_deg, indices = shop_tree.query(school_coords)
    
    # 使用Haversine公式计算精确距离
    distances_meters = []
    for i, idx in enumerate(indices):
        dist_m = haversine_distance(
            school_coords[i][0], school_coords[i][1],
            shop_coords[idx][0], shop_coords[idx][1]
        )
        distances_meters.append(dist_m)
    
    distances_meters = np.array(distances_meters)
    
    # 添加距离信息到学校GeoDataFrame
    school_gdf['nearest_shop_distance_m'] = distances_meters
    school_gdf['nearest_shop_index'] = indices
    
    # 获取最近商店的信息
    nearest_shop_names = shop_gdf.iloc[indices][shop_gdf.columns[0]].values
    school_gdf['nearest_shop'] = nearest_shop_names
    
    print(f"距离计算完成，最小距离: {distances_meters.min():.2f}米, 最大距离: {distances_meters.max():.2f}米")
    print(f"平均距离: {distances_meters.mean():.2f}米")
    
    return school_gdf, shop_gdf

# 执行距离计算
try:
    school_with_accessibility, shop_gdf = calculate_shop_accessibility(shop_gdf, school_gdf)
except ValueError as e:
    print(f"计算可达性时出错: {str(e)}")
    raise

# 6. 创建栅格化函数
def create_shop_accessibility_grid(school_gdf, shop_gdf, cell_size=10, buffer_distance=5000):  # 修改默认值为10米
    """
    将学校到商店的可达性数据转换为栅格
    使用反距离加权插值方法
    """
    print("开始创建栅格数据...")
    
    if len(school_gdf) == 0 or len(shop_gdf) == 0:
        raise ValueError("学校或商店数据为空，无法创建栅格")
    
    # 获取研究区域边界
    school_bounds = school_gdf.total_bounds  # (minx, miny, maxx, maxy)
    shop_bounds = shop_gdf.total_bounds
    
    # 综合考虑学校和商店的边界
    minx = min(school_bounds[0], shop_bounds[0])
    miny = min(school_bounds[1], shop_bounds[1])
    maxx = max(school_bounds[2], shop_bounds[2])
    maxy = max(school_bounds[3], shop_bounds[3])
    
    # 扩展边界以包含缓冲区
    lat_factor = np.cos(np.radians((miny + maxy) / 2))
    minx -= buffer_distance / (111000 * lat_factor)
    miny -= buffer_distance / 111000
    maxx += buffer_distance / (111000 * lat_factor)
    maxy += buffer_distance / 111000
    
    # 计算栅格尺寸（使用Haversine距离转换）
    x_range = haversine_distance(minx, (miny + maxy) / 2, maxx, (miny + maxy) / 2)
    y_range = haversine_distance((minx + maxx) / 2, miny, (minx + maxx) / 2, maxy)
    
    width = max(10, int(x_range / cell_size))  # 确保至少有10个像素
    height = max(10, int(y_range / cell_size))
    
    print(f"研究区域范围: ({minx:.6f}, {miny:.6f}, {maxx:.6f}, {maxy:.6f})")
    print(f"栅格尺寸: {width} x {height}, 像素大小: {cell_size}米")
    
    # 创建坐标网格
    x_coords = np.linspace(minx, maxx, width)
    y_coords = np.linspace(maxy, miny, height)  # 注意：y轴从上到下
    xx, yy = np.meshgrid(x_coords, y_coords)
    
    # 展平坐标用于距离计算
    grid_points = np.column_stack([xx.ravel(), yy.ravel()])
    
    # 提取学校坐标和距离值
    school_coords = np.array([[point.x, point.y] for point in school_gdf.geometry])
    school_distances = school_gdf['nearest_shop_distance_m'].values
    
    # 创建KDTree
    school_tree = cKDTree(school_coords)
    
    # 对于每个栅格点，计算插值
    accessibility_grid = np.full(len(grid_points), np.nan)
    
    # 设置插值参数
    k_neighbors = min(10, len(school_coords))  # 使用最近的10个点
    power = 2  # IDW的幂参数
    
    print("正在进行空间插值...")
    print(f"总栅格点数: {len(grid_points)} (栅格尺寸: {height} x {width})")
    
    # 为了处理大数据集，我们可以分批处理
    batch_size = 10000
    for start_idx in range(0, len(grid_points), batch_size):
        end_idx = min(start_idx + batch_size, len(grid_points))
        
        for i in range(start_idx, end_idx):
            if i % 1000 == 0:  # 每处理1000个点打印一次进度
                print(f"已处理 {i}/{len(grid_points)} 个栅格点")
            
            # 找到最近的k个学校点
            distances_to_schools, indices = school_tree.query(grid_points[i], k=k_neighbors)
            
            # 处理单个点的情况
            if np.isscalar(distances_to_schools):
                distances_to_schools = np.array([distances_to_schools])
                indices = np.array([indices])
            
            # 计算加权平均
            weights = []
            values = []
            
            for j, (dist, idx) in enumerate(zip(distances_to_schools, indices)):
                if dist < 1e-6:  # 如果距离非常小，直接使用该点的值
                    accessibility_grid[i] = school_distances[idx]
                    break
                else:
                    # 使用反距离加权
                    weight = 1 / (dist ** power)
                    weights.append(weight)
                    values.append(school_distances[idx])
            else:
                # 如果没有距离为0的点，则计算加权平均
                if len(weights) > 0:
                    total_weight = sum(weights)
                    if total_weight > 0:
                        accessibility_grid[i] = sum(w * v for w, v in zip(weights, values)) / total_weight
                    else:
                        accessibility_grid[i] = np.nan
                else:
                    accessibility_grid[i] = np.nan
    
    # 重塑为2D栅格
    accessibility_grid = accessibility_grid.reshape(height, width)
    
    # 计算地理变换参数
    x_pixel_size = (maxx - minx) / width
    y_pixel_size = (maxy - miny) / height  # 注意这里是负值，因为y轴从上到下
    transform = from_origin(minx, maxy, x_pixel_size, abs(y_pixel_size))
    
    print("栅格数据创建完成")
    return accessibility_grid, transform, (minx, miny, maxx, maxy)

# 创建栅格数据 - 修改为10米精度
try:
    accessibility_grid, transform, bounds = create_shop_accessibility_grid(school_with_accessibility, shop_gdf, cell_size=10)  # 改为10米
except Exception as e:
    print(f"创建栅格时出错: {str(e)}")
    raise

# 7. 输出栅格数据
def save_raster_data(grid, transform, bounds, output_path):
    """
    保存栅格数据为GeoTIFF格式
    """
    print(f"开始保存栅格数据到: {output_path}")
    
    # 确保输出目录存在
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    with rasterio.open(
        output_path,
        'w',
        driver='GTiff',
        height=grid.shape[0],
        width=grid.shape[1],
        count=1,
        dtype=rasterio.float32,
        crs='EPSG:4326',
        transform=transform,
        nodata=-9999
    ) as dst:
        # 将NaN值替换为nodata值
        grid_masked = np.where(np.isnan(grid), -9999, grid).astype(np.float32)
        dst.write(grid_masked, 1)
    
    print("栅格数据保存完成")

# 保存可达性栅格
output_raster_path = r'E:\chuli\accessibility\school_to_shop_accessibility_10m.tif'  # 修改文件名
try:
    save_raster_data(accessibility_grid, transform, bounds, output_raster_path)
except Exception as e:
    print(f"保存栅格数据时出错: {str(e)}")
    raise

# 8. 可视化结果
def visualize_results(school_gdf, shop_gdf, accessibility_grid, bounds, save_path=None):
    """
    可视化学校到商店的可达性分析结果
    """
    print("开始可视化...")
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    # 左图：原始数据分布
    ax1 = axes[0]
    ax1.scatter(shop_gdf.geometry.x, shop_gdf.geometry.y, c='green', s=1, alpha=0.6, label='商店')
    ax1.scatter(school_gdf.geometry.x, school_gdf.geometry.y, c='red', s=20, alpha=0.8, label='学校')
    ax1.set_title('学校与商店分布')
    ax1.set_xlabel('经度')
    ax1.set_ylabel('纬度')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 右图：可达性栅格
    ax2 = axes[1]
    # 过滤掉NaN值用于可视化
    valid_data = accessibility_grid[~np.isnan(accessibility_grid)]
    if len(valid_data) > 0:
        vmax = np.nanpercentile(accessibility_grid, 95)
        im = ax2.imshow(accessibility_grid, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], 
                        cmap='RdYlBu_r', origin='upper', vmin=0, vmax=vmax)
    else:
        im = ax2.imshow(accessibility_grid, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], 
                        cmap='RdYlBu_r', origin='upper')
    
    ax2.scatter(school_gdf.geometry.x, school_gdf.geometry.y, c='black', s=10, alpha=0.8, label='学校')
    ax2.set_title('学校到最近商店距离（米）')
    ax2.set_xlabel('经度')
    ax2.set_ylabel('纬度')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.colorbar(im, ax=ax2, label='距离（米）', shrink=0.8)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"可视化结果已保存到: {save_path}")
    
    plt.show()

# 可视化结果
viz_path = r'E:\chuli\accessibility\school_shop_accessibility_visualization_10m.png'  # 修改文件名
os.makedirs(os.path.dirname(viz_path), exist_ok=True)
try:
    visualize_results(school_with_accessibility, shop_gdf, accessibility_grid, bounds, viz_path)
except Exception as e:
    print(f"可视化时出错: {str(e)}")

# 9. 生成统计报告
def generate_statistics(school_gdf):
    """
    生成学校到商店的可达性统计报告
    """
    distances = school_gdf['nearest_shop_distance_m'].dropna()
    
    if len(distances) == 0:
        print("警告: 没有有效的距离数据")
        return {}
    
    stats = {
        '学校总数': len(school_gdf),
        '有商店可达的学校数': len(distances),
        '平均距离（米）': distances.mean(),
        '中位数距离（米）': distances.median(),
        '最短距离（米）': distances.min(),
        '最长距离（米）': distances.max(),
        '标准差（米）': distances.std(),
        '距离小于500米的学校数': (distances < 500).sum(),
        '距离小于1000米的学校数': (distances < 1000).sum(),
        '距离小于2000米的学校数': (distances < 2000).sum(),
        '距离小于3000米的学校数': (distances < 3000).sum()
    }
    
    print("\n=== 学校到商店可达性统计报告 ===")
    for key, value in stats.items():
        if isinstance(value, float):
            print(f"{key}: {value:.2f}")
        else:
            print(f"{key}: {value}")
    
    return stats

# 生成统计报告
stats = generate_statistics(school_with_accessibility)

# 10. 保存处理后的学校数据
school_output_path = r'E:\chuli\accessibility\school_to_shop_accessibility_10m.csv'  # 修改文件名
try:
    school_with_accessibility.to_csv(school_output_path, index=False, encoding='utf-8-sig')
    print(f"\n学校到商店可达性数据已保存到: {school_output_path}")
except Exception as e:
    print(f"保存学校数据时出错: {str(e)}")

# 11. 创建不同距离阈值的二进制栅格
def create_binary_accessibility_grids(accessibility_grid, transform, bounds, thresholds=[500, 1000, 2000]):
    """
    创建不同距离阈值的二进制可达性栅格
    """
    print("创建不同距离阈值的二进制栅格...")
    
    for threshold in thresholds:
        # 创建二进制栅格（可达=1，不可达=0），处理NaN值
        binary_grid = np.where(np.isnan(accessibility_grid), 0, 
                              np.where(accessibility_grid <= threshold, 1, 0)).astype(np.int8)
        
        # 保存二进制栅格
        binary_path = f'E:\\chuli\\accessibility\\school_to_shop_accessibility_10m_within_{threshold}m.tif'
        try:
            save_raster_data(binary_grid, transform, bounds, binary_path)
            print(f"距离阈值{threshold}米的二进制栅格已保存: {binary_path}")
        except Exception as e:
            print(f"保存二进制栅格 {binary_path} 时出错: {str(e)}")

# 创建二进制栅格
create_binary_accessibility_grids(accessibility_grid, transform, bounds, [500, 1000, 2000])

print(f"\n=== 分析完成 ===")
print(f"栅格文件位置: {output_raster_path}")
print(f"可视化文件位置: {viz_path}")
print(f"学校数据文件位置: {school_output_path}")

# 12. 显示栅格基本信息
print(f"\n=== 栅格数据信息 ===")
print(f"栅格尺寸: {accessibility_grid.shape}")
valid_data = accessibility_grid[~np.isnan(accessibility_grid)]
if len(valid_data) > 0:
    print(f"有效数据范围: {np.nanmin(accessibility_grid):.2f} - {np.nanmax(accessibility_grid):.2f} 米")
print(f"NaN值数量: {np.count_nonzero(np.isnan(accessibility_grid))}")
print(f"有效值数量: {np.count_nonzero(~np.isnan(accessibility_grid))}")

print("\n所有处理完成！")
print("学校到商店的可达性分析已完成，结果保存在 E:\\chuli\\accessibility 文件夹中。")
print("栅格精度为10米×10米。")


In [ ]:
# 导入必要的库
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import os
import rasterio
from rasterio.transform import from_origin
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("开始加载纽约数据...")

# 1. 加载数据 - 尝试不同的编码格式
def load_csv_with_encoding(file_path):
    """
    尝试多种编码格式加载CSV文件
    """
    encodings = ['utf-8', 'gbk', 'gb2312', 'latin-1', 'cp1252']
    
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"成功使用 {encoding} 编码加载文件: {file_path}")
            return df
        except UnicodeDecodeError:
            print(f"编码 {encoding} 失败，尝试下一个...")
            continue
        except Exception as e:
            print(f"编码 {encoding} 出现其他错误: {str(e)}")
            continue
    
    raise Exception(f"无法使用任何编码格式加载文件: {file_path}")

# 加载数据
try:
    shop_data = load_csv_with_encoding(r'E:\chuli\traffic\NY_shop_cut.csv')
    bus_data = load_csv_with_encoding(r'E:\chuli\traffic\NY_busStation_cut.csv')
    school_data = load_csv_with_encoding(r'E:\chuli\over\school_NY.csv')
except Exception as e:
    print(f"加载数据时出错: {str(e)}")
    print("请检查文件路径和文件格式")
    raise

print(f"商店数据形状: {shop_data.shape}")
print(f"公交站数据形状: {bus_data.shape}")
print(f"学校数据形状: {school_data.shape}")

# 检查列名
print("商店数据列名:", shop_data.columns.tolist())
print("公交站数据列名:", bus_data.columns.tolist())
print("学校数据列名:", school_data.columns.tolist())

# 2. 数据预处理 - 假设经纬度列名
def find_coord_columns(df, data_type):
    """自动查找坐标列"""
    lon_patterns = ['longitude', 'lon', 'lng', 'x', 'X', 'LONG', 'LON', '经度', 'x坐标', 'X坐标', 'start_lon', 'end_lon', 'lng_start', 'lng_end']
    lat_patterns = ['latitude', 'lat', 'y', 'Y', 'LAT', '纬度', 'y坐标', 'Y坐标', 'start_lat', 'end_lat', 'lat_start', 'lat_end']
    
    lon_col = None
    lat_col = None
    
    for pattern in lon_patterns:
        for col in df.columns:
            if pattern.lower() in col.lower():
                lon_col = col
                break
        if lon_col:
            break
    
    for pattern in lat_patterns:
        for col in df.columns:
            if pattern.lower() in col.lower():
                lat_col = col
                break
        if lat_col:
            break
    
    if lon_col is None or lat_col is None:
        # 如果没找到标准列名，使用前两列
        cols = df.columns.tolist()
        if len(cols) >= 2:
            lon_col, lat_col = cols[0], cols[1]
            print(f"警告: {data_type}数据未找到标准坐标列名，使用前两列: {lon_col}, {lat_col}")
        else:
            raise ValueError(f"{data_type}数据列数不足，至少需要2列坐标数据")
    
    return lon_col, lat_col

try:
    shop_lon_col, shop_lat_col = find_coord_columns(shop_data, "商店")
    bus_lon_col, bus_lat_col = find_coord_columns(bus_data, "公交站")
    school_lon_col, school_lat_col = find_coord_columns(school_data, "学校")
except ValueError as e:
    print(f"数据列名错误: {str(e)}")
    print("请确保CSV文件包含经纬度坐标列")
    raise

print(f"商店经度列: {shop_lon_col}, 纬度列: {shop_lat_col}")
print(f"公交站经度列: {bus_lon_col}, 纬度列: {bus_lat_col}")
print(f"学校经度列: {school_lon_col}, 纬度列: {school_lat_col}")

# 3. 数据清洗和验证
def clean_coordinate_data(df, lon_col, lat_col, data_type):
    """
    清洗坐标数据，移除无效值
    """
    print(f"清洗{data_type}坐标数据...")
    
    # 检查并转换坐标数据类型
    try:
        df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
        df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    except Exception as e:
        print(f"转换{data_type}坐标数据类型时出错: {str(e)}")
        raise
    
    # 移除无效坐标（NaN, 无穷大等）
    original_count = len(df)
    df = df.dropna(subset=[lon_col, lat_col])
    
    # 检查坐标范围（纽约的经纬度范围：经度-74到-73，纬度40到41）
    df = df[(df[lon_col] >= -75) & (df[lon_col] <= -72) & 
            (df[lat_col] >= 40) & (df[lat_col] <= 42)]
    
    removed_count = original_count - len(df)
    print(f"{data_type}数据清洗完成: 原始{original_count}条，移除{removed_count}条无效数据，剩余{len(df)}条")
    
    return df

# 清洗数据
shop_data = clean_coordinate_data(shop_data, shop_lon_col, shop_lat_col, "商店")
bus_data = clean_coordinate_data(bus_data, bus_lon_col, bus_lat_col, "公交站")
school_data = clean_coordinate_data(school_data, school_lon_col, school_lat_col, "学校")

# 4. 创建GeoDataFrame
shop_geometry = [Point(xy) for xy in zip(shop_data[shop_lon_col], shop_data[shop_lat_col])]
shop_gdf = gpd.GeoDataFrame(shop_data, geometry=shop_geometry, crs='EPSG:4326')

bus_geometry = [Point(xy) for xy in zip(bus_data[bus_lon_col], bus_data[bus_lat_col])]
bus_gdf = gpd.GeoDataFrame(bus_data, geometry=bus_geometry, crs='EPSG:4326')

school_geometry = [Point(xy) for xy in zip(school_data[school_lon_col], school_data[school_lat_col])]
school_gdf = gpd.GeoDataFrame(school_data, geometry=school_geometry, crs='EPSG:4326')

print("数据加载和清洗完成")

# 5. 计算距离的通用函数
def haversine_distance(lon1, lat1, lon2, lat2):
    """
    使用Haversine公式计算两点间的距离（米）
    """
    # 转换为弧度
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    
    # Haversine公式
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    # 地球半径（米）
    r = 6371000
    
    return c * r

def calculate_accessibility(facility_gdf, school_gdf, facility_name):
    """
    计算学校到最近设施的距离
    使用KDTree进行快速最近邻搜索
    """
    print(f"开始计算学校到最近{facility_name}的距离...")
    
    if len(facility_gdf) == 0 or len(school_gdf) == 0:
        raise ValueError(f"{facility_name}或学校数据为空，无法计算可达性")
    
    # 提取坐标
    facility_coords = np.array([[point.x, point.y] for point in facility_gdf.geometry])
    school_coords = np.array([[point.x, point.y] for point in school_gdf.geometry])
    
    # 使用KDTree进行最近邻搜索
    facility_tree = cKDTree(facility_coords)
    
    # 计算每个学校到最近设施的距离
    distances_deg, indices = facility_tree.query(school_coords)
    
    # 使用Haversine公式计算精确距离
    distances_meters = []
    for i, idx in enumerate(indices):
        dist_m = haversine_distance(
            school_coords[i][0], school_coords[i][1],
            facility_coords[idx][0], facility_coords[idx][1]
        )
        distances_meters.append(dist_m)
    
    distances_meters = np.array(distances_meters)
    
    # 添加距离信息到学校GeoDataFrame
    distance_col_name = f'nearest_{facility_name.lower()}_distance_m'
    index_col_name = f'nearest_{facility_name.lower()}_index'
    
    school_gdf[distance_col_name] = distances_meters
    school_gdf[index_col_name] = indices
    
    # 获取最近设施的信息
    nearest_facility_names = facility_gdf.iloc[indices][facility_gdf.columns[0]].values
    school_gdf[f'nearest_{facility_name.lower()}'] = nearest_facility_names
    
    print(f"{facility_name}距离计算完成，最小距离: {distances_meters.min():.2f}米, 最大距离: {distances_meters.max():.2f}米")
    print(f"平均距离: {distances_meters.mean():.2f}米")
    
    return school_gdf, facility_gdf

# 计算学校到商店和公交站的可达性
try:
    school_gdf, shop_gdf = calculate_accessibility(shop_gdf, school_gdf, "Shop")
    school_gdf, bus_gdf = calculate_accessibility(bus_gdf, school_gdf, "Bus")
except ValueError as e:
    print(f"计算可达性时出错: {str(e)}")
    raise

# 6. 创建栅格化函数
def create_accessibility_grid(school_gdf, facility_gdf, cell_size=30, buffer_distance=5000):
    """
    将学校可达性数据转换为栅格
    使用反距离加权插值方法
    """
    print("开始创建栅格数据...")
    
    if len(school_gdf) == 0 or len(facility_gdf) == 0:
        raise ValueError("学校或设施数据为空，无法创建栅格")
    
    # 获取研究区域边界
    school_bounds = school_gdf.total_bounds  # (minx, miny, maxx, maxy)
    facility_bounds = facility_gdf.total_bounds
    
    # 综合考虑学校和设施的边界
    minx = min(school_bounds[0], facility_bounds[0])
    miny = min(school_bounds[1], facility_bounds[1])
    maxx = max(school_bounds[2], facility_bounds[2])
    maxy = max(school_bounds[3], facility_bounds[3])
    
    # 扩展边界以包含缓冲区
    lat_factor = np.cos(np.radians((miny + maxy) / 2))
    minx -= buffer_distance / (111000 * lat_factor)
    miny -= buffer_distance / 111000
    maxx += buffer_distance / (111000 * lat_factor)
    maxy += buffer_distance / 111000
    
    # 计算栅格尺寸（使用Haversine距离转换）
    x_range = haversine_distance(minx, (miny + maxy) / 2, maxx, (miny + maxy) / 2)
    y_range = haversine_distance((minx + maxx) / 2, miny, (minx + maxx) / 2, maxy)
    
    width = max(10, int(x_range / cell_size))  # 确保至少有10个像素
    height = max(10, int(y_range / cell_size))
    
    print(f"研究区域范围: ({minx:.6f}, {miny:.6f}, {maxx:.6f}, {maxy:.6f})")
    print(f"栅格尺寸: {width} x {height}, 像素大小: {cell_size}米")
    
    # 创建坐标网格
    x_coords = np.linspace(minx, maxx, width)
    y_coords = np.linspace(maxy, miny, height)  # 注意：y轴从上到下
    xx, yy = np.meshgrid(x_coords, y_coords)
    
    # 展平坐标用于距离计算
    grid_points = np.column_stack([xx.ravel(), yy.ravel()])
    
    # 提取学校坐标和距离值
    school_coords = np.array([[point.x, point.y] for point in school_gdf.geometry])
    
    # 获取对应的距离列（查找包含distance_m的列）
    distance_cols = [col for col in school_gdf.columns if 'distance_m' in col]
    if len(distance_cols) < 2:
        raise ValueError("没有找到足够的距离列")
    
    # 分别处理商店和公交站的距离
    shop_distance_col = [col for col in distance_cols if 'shop' in col.lower()][0]
    bus_distance_col = [col for col in distance_cols if 'bus' in col.lower()][0]
    
    shop_distances = school_gdf[shop_distance_col].values
    bus_distances = school_gdf[bus_distance_col].values
    
    print(f"商店距离列: {shop_distance_col}")
    print(f"公交站距离列: {bus_distance_col}")
    
    # 为商店和公交站分别创建栅格
    accessibility_grids = {}
    
    for name, distances in [("shop", shop_distances), ("bus", bus_distances)]:
        print(f"正在创建{name}可达性栅格...")
        
        # 创建KDTree
        school_tree = cKDTree(school_coords)
        
        # 对于每个栅格点，计算插值
        accessibility_grid = np.full(len(grid_points), np.nan)
        
        # 设置插值参数
        k_neighbors = min(10, len(school_coords))  # 使用最近的10个点
        power = 2  # IDW的幂参数
        
        print(f"正在进行{name}空间插值...")
        print(f"总栅格点数: {len(grid_points)} (栅格尺寸: {height} x {width})")
        
        # 分批处理以节省内存
        batch_size = 5000
        for start_idx in range(0, len(grid_points), batch_size):
            end_idx = min(start_idx + batch_size, len(grid_points))
            
            for i in range(start_idx, end_idx):
                if i % 1000 == 0:  # 每处理1000个点打印一次进度
                    print(f"已处理 {i}/{len(grid_points)} 个栅格点 ({name})")
                
                # 找到最近的k个学校点
                distances_to_schools, indices = school_tree.query(grid_points[i], k=k_neighbors)
                
                # 处理单个点的情况
                if np.isscalar(distances_to_schools):
                    distances_to_schools = np.array([distances_to_schools])
                    indices = np.array([indices])
                
                # 计算加权平均
                weights = []
                values = []
                
                for j, (dist, idx) in enumerate(zip(distances_to_schools, indices)):
                    if dist < 1e-6:  # 如果距离非常小，直接使用该点的值
                        accessibility_grid[i] = distances[idx]
                        break
                    else:
                        # 使用反距离加权
                        weight = 1 / (dist ** power)
                        weights.append(weight)
                        values.append(distances[idx])
                else:
                    # 如果没有距离为0的点，则计算加权平均
                    if len(weights) > 0:
                        total_weight = sum(weights)
                        if total_weight > 0:
                            accessibility_grid[i] = sum(w * v for w, v in zip(weights, values)) / total_weight
                        else:
                            accessibility_grid[i] = np.nan
                    else:
                        accessibility_grid[i] = np.nan
        
        # 重塑为2D栅格
        accessibility_grid = accessibility_grid.reshape(height, width)
        accessibility_grids[name] = accessibility_grid
    
    # 计算地理变换参数
    x_pixel_size = (maxx - minx) / width
    y_pixel_size = (maxy - miny) / height  # 注意这里是负值，因为y轴从上到下
    transform = from_origin(minx, maxy, x_pixel_size, abs(y_pixel_size))
    
    print("栅格数据创建完成")
    return accessibility_grids, transform, (minx, miny, maxx, maxy)

# 创建栅格数据
try:
    accessibility_grids, transform, bounds = create_accessibility_grid(school_gdf, shop_gdf, cell_size=30)  # 30米精度
except Exception as e:
    print(f"创建栅格时出错: {str(e)}")
    raise

# 7. 输出栅格数据
def save_raster_data(grid, transform, bounds, output_path):
    """
    保存栅格数据为GeoTIFF格式
    """
    print(f"开始保存栅格数据到: {output_path}")
    
    # 确保输出目录存在
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    with rasterio.open(
        output_path,
        'w',
        driver='GTiff',
        height=grid.shape[0],
        width=grid.shape[1],
        count=1,
        dtype=rasterio.float32,
        crs='EPSG:4326',
        transform=transform,
        nodata=-9999
    ) as dst:
        # 将NaN值替换为nodata值
        grid_masked = np.where(np.isnan(grid), -9999, grid).astype(np.float32)
        dst.write(grid_masked, 1)
    
    print("栅格数据保存完成")

# 保存可达性栅格
for facility_type, grid in accessibility_grids.items():
    output_raster_path = f'E:\\chuli\\accessibility\\school_to_{facility_type}_accessibility_30m.tif'
    try:
        save_raster_data(grid, transform, bounds, output_raster_path)
        print(f"{facility_type}可达性栅格已保存: {output_raster_path}")
    except Exception as e:
        print(f"保存{facility_type}栅格数据时出错: {str(e)}")

# 8. 可视化结果
def visualize_results(school_gdf, shop_gdf, bus_gdf, accessibility_grids, bounds, save_path=None):
    """
    可视化学校到设施的可达性分析结果
    """
    print("开始可视化...")
    
    fig, axes = plt.subplots(2, 2, figsize=(20, 16))
    
    # 左上图：原始数据分布
    ax1 = axes[0, 0]
    ax1.scatter(shop_gdf.geometry.x, shop_gdf.geometry.y, c='green', s=1, alpha=0.6, label='商店')
    ax1.scatter(bus_gdf.geometry.x, bus_gdf.geometry.y, c='blue', s=1, alpha=0.6, label='公交站')
    ax1.scatter(school_gdf.geometry.x, school_gdf.geometry.y, c='red', s=20, alpha=0.8, label='学校')
    ax1.set_title('学校、商店与公交站分布')
    ax1.set_xlabel('经度')
    ax1.set_ylabel('纬度')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 右上图：学校到商店可达性
    ax2 = axes[0, 1]
    shop_grid = accessibility_grids['shop']
    valid_data = shop_grid[~np.isnan(shop_grid)]
    if len(valid_data) > 0:
        vmax = np.nanpercentile(shop_grid, 95)
        im1 = ax2.imshow(shop_grid, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], 
                        cmap='RdYlBu_r', origin='upper', vmin=0, vmax=vmax)
    else:
        im1 = ax2.imshow(shop_grid, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], 
                        cmap='RdYlBu_r', origin='upper')
    
    ax2.scatter(school_gdf.geometry.x, school_gdf.geometry.y, c='black', s=10, alpha=0.8, label='学校')
    ax2.set_title('学校到最近商店距离（米）')
    ax2.set_xlabel('经度')
    ax2.set_ylabel('纬度')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.colorbar(im1, ax=ax2, label='距离（米）', shrink=0.8)
    
    # 左下图：学校到公交站可达性
    ax3 = axes[1, 0]
    bus_grid = accessibility_grids['bus']
    valid_data = bus_grid[~np.isnan(bus_grid)]
    if len(valid_data) > 0:
        vmax = np.nanpercentile(bus_grid, 95)
        im2 = ax3.imshow(bus_grid, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], 
                        cmap='RdYlBu_r', origin='upper', vmin=0, vmax=vmax)
    else:
        im2 = ax3.imshow(bus_grid, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], 
                        cmap='RdYlBu_r', origin='upper')
    
    ax3.scatter(school_gdf.geometry.x, school_gdf.geometry.y, c='black', s=10, alpha=0.8, label='学校')
    ax3.set_title('学校到最近公交站距离（米）')
    ax3.set_xlabel('经度')
    ax3.set_ylabel('纬度')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    plt.colorbar(im2, ax=ax3, label='距离（米）', shrink=0.8)
    
    # 右下图：综合可达性（商店+公交站平均距离）
    ax4 = axes[1, 1]
    combined_grid = (accessibility_grids['shop'] + accessibility_grids['bus']) / 2
    valid_data = combined_grid[~np.isnan(combined_grid)]
    if len(valid_data) > 0:
        vmax = np.nanpercentile(combined_grid, 95)
        im3 = ax4.imshow(combined_grid, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], 
                        cmap='RdYlBu_r', origin='upper', vmin=0, vmax=vmax)
    else:
        im3 = ax4.imshow(combined_grid, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], 
                        cmap='RdYlBu_r', origin='upper')
    
    ax4.scatter(school_gdf.geometry.x, school_gdf.geometry.y, c='black', s=10, alpha=0.8, label='学校')
    ax4.set_title('学校综合可达性（商店+公交站平均距离）（米）')
    ax4.set_xlabel('经度')
    ax4.set_ylabel('纬度')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.colorbar(im3, ax=ax4, label='距离（米）', shrink=0.8)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"可视化结果已保存到: {save_path}")
    
    plt.show()

# 可视化结果
viz_path = r'E:\chuli\accessibility\school_accessibility_visualization_30m.png'
os.makedirs(os.path.dirname(viz_path), exist_ok=True)
try:
    visualize_results(school_gdf, shop_gdf, bus_gdf, accessibility_grids, bounds, viz_path)
except Exception as e:
    print(f"可视化时出错: {str(e)}")

# 9. 生成统计报告
def generate_statistics(school_gdf):
    """
    生成学校可达性统计报告
    """
    # 获取距离列
    distance_cols = [col for col in school_gdf.columns if 'distance_m' in col]
    if len(distance_cols) < 2:
        print("警告: 没有找到足够的距离列")
        return {}
    
    shop_distance_col = [col for col in distance_cols if 'shop' in col.lower()][0]
    bus_distance_col = [col for col in distance_cols if 'bus' in col.lower()][0]
    
    shop_distances = school_gdf[shop_distance_col].dropna()
    bus_distances = school_gdf[bus_distance_col].dropna()
    
    if len(shop_distances) == 0 or len(bus_distances) == 0:
        print("警告: 没有有效的距离数据")
        return {}
    
    stats = {
        '学校总数': len(school_gdf),
        '有商店可达的学校数': len(shop_distances),
        '有公交站可达的学校数': len(bus_distances),
        '学校到商店平均距离（米）': shop_distances.mean(),
        '学校到公交站平均距离（米）': bus_distances.mean(),
        '学校到商店中位数距离（米）': shop_distances.median(),
        '学校到公交站中位数距离（米）': bus_distances.median(),
        '学校到商店最短距离（米）': shop_distances.min(),
        '学校到公交站最短距离（米）': bus_distances.min(),
        '学校到商店最长距离（米）': shop_distances.max(),
        '学校到公交站最长距离（米）': bus_distances.max(),
        '学校到商店标准差（米）': shop_distances.std(),
        '学校到公交站标准差（米）': bus_distances.std(),
        '到商店距离<500米的学校数': (shop_distances < 500).sum(),
        '到公交站距离<500米的学校数': (bus_distances < 500).sum(),
        '到商店距离<1000米的学校数': (shop_distances < 1000).sum(),
        '到公交站距离<1000米的学校数': (bus_distances < 1000).sum(),
    }
    
    print("\n=== 学校可达性统计报告 ===")
    for key, value in stats.items():
        if isinstance(value, float):
            print(f"{key}: {value:.2f}")
        else:
            print(f"{key}: {value}")
    
    return stats

# 生成统计报告
stats = generate_statistics(school_gdf)

# 10. 保存处理后的学校数据
school_output_path = r'E:\chuli\accessibility\school_accessibility_30m.csv'
try:
    school_gdf.to_csv(school_output_path, index=False, encoding='utf-8-sig')
    print(f"\n学校可达性数据已保存到: {school_output_path}")
except Exception as e:
    print(f"保存学校数据时出错: {str(e)}")

# 11. 创建不同距离阈值的二进制栅格
def create_binary_accessibility_grids(accessibility_grids, transform, bounds, thresholds=[500, 1000, 2000]):
    """
    创建不同距离阈值的二进制可达性栅格
    """
    print("创建不同距离阈值的二进制栅格...")
    
    for facility_type, grid in accessibility_grids.items():
        for threshold in thresholds:
            # 创建二进制栅格（可达=1，不可达=0），处理NaN值
            binary_grid = np.where(np.isnan(grid), 0, 
                                  np.where(grid <= threshold, 1, 0)).astype(np.int8)
            
            # 保存二进制栅格
            binary_path = f'E:\\chuli\\accessibility\\school_to_{facility_type}_accessibility_30m_within_{threshold}m.tif'
            try:
                save_raster_data(binary_grid, transform, bounds, binary_path)
                print(f"{facility_type}距离阈值{threshold}米的二进制栅格已保存: {binary_path}")
            except Exception as e:
                print(f"保存二进制栅格 {binary_path} 时出错: {str(e)}")

# 创建二进制栅格
create_binary_accessibility_grids(accessibility_grids, transform, bounds, [500, 1000, 2000])

print(f"\n=== 分析完成 ===")
print("所有栅格文件、可视化图像和统计数据已保存到 E:\\chuli\\accessibility 文件夹中")
print("包含以下文件：")
for facility_type in accessibility_grids.keys():
    print(f"- 学校到{facility_type}可达性栅格 (30m精度)")
    print(f"- 学校到{facility_type}不同距离阈值的二进制栅格")

# 12. 显示栅格基本信息
print(f"\n=== 栅格数据信息 ===")
for facility_type, grid in accessibility_grids.items():
    print(f"{facility_type}栅格尺寸: {grid.shape}")
    valid_data = grid[~np.isnan(grid)]
    if len(valid_data) > 0:
        print(f"{facility_type}有效数据范围: {np.nanmin(grid):.2f} - {np.nanmax(grid):.2f} 米")
    print(f"{facility_type}NaN值数量: {np.count_nonzero(np.isnan(grid))}")
    print(f"{facility_type}有效值数量: {np.count_nonzero(~np.isnan(grid))}")

print("\n所有处理完成！")
print("纽约学校到商店和公交站的可达性分析已完成。")
print("栅格精度为30米×30米。")


In [ ]:
# 导入必要的库
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import os
import rasterio
from rasterio.transform import from_origin
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("开始加载纽约数据...")

# 1. 加载数据 - 尝试不同的编码格式
def load_csv_with_encoding(file_path):
    """
    尝试多种编码格式加载CSV文件
    """
    encodings = ['utf-8', 'gbk', 'gb2312', 'latin-1', 'cp1252']
    
    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"成功使用 {encoding} 编码加载文件: {file_path}")
            return df
        except UnicodeDecodeError:
            print(f"编码 {encoding} 失败，尝试下一个...")
            continue
        except Exception as e:
            print(f"编码 {encoding} 出现其他错误: {str(e)}")
            continue
    
    raise Exception(f"无法使用任何编码格式加载文件: {file_path}")

# 加载数据
try:
    shop_data = load_csv_with_encoding(r'E:\chuli\traffic\NY_shop_cut.csv')
    bus_data = load_csv_with_encoding(r'E:\chuli\traffic\NY_busStation_cut.csv')
    school_data = load_csv_with_encoding(r'E:\chuli\over\school_NY.csv')
except Exception as e:
    print(f"加载数据时出错: {str(e)}")
    print("请检查文件路径和文件格式")
    raise

print(f"商店数据形状: {shop_data.shape}")
print(f"公交站数据形状: {bus_data.shape}")
print(f"学校数据形状: {school_data.shape}")

# 检查列名
print("商店数据列名:", shop_data.columns.tolist())
print("公交站数据列名:", bus_data.columns.tolist())
print("学校数据列名:", school_data.columns.tolist())

# 2. 数据预处理 - 假设经纬度列名
def find_coord_columns(df, data_type):
    """自动查找坐标列"""
    lon_patterns = ['longitude', 'lon', 'lng', 'x', 'X', 'LONG', 'LON', '经度', 'x坐标', 'X坐标', 'start_lon', 'end_lon', 'lng_start', 'lng_end']
    lat_patterns = ['latitude', 'lat', 'y', 'Y', 'LAT', '纬度', 'y坐标', 'Y坐标', 'start_lat', 'end_lat', 'lat_start', 'lat_end']
    
    lon_col = None
    lat_col = None
    
    for pattern in lon_patterns:
        for col in df.columns:
            if pattern.lower() in col.lower():
                lon_col = col
                break
        if lon_col:
            break
    
    for pattern in lat_patterns:
        for col in df.columns:
            if pattern.lower() in col.lower():
                lat_col = col
                break
        if lat_col:
            break
    
    if lon_col is None or lat_col is None:
        # 如果没找到标准列名，使用前两列
        cols = df.columns.tolist()
        if len(cols) >= 2:
            lon_col, lat_col = cols[0], cols[1]
            print(f"警告: {data_type}数据未找到标准坐标列名，使用前两列: {lon_col}, {lat_col}")
        else:
            raise ValueError(f"{data_type}数据列数不足，至少需要2列坐标数据")
    
    return lon_col, lat_col

try:
    shop_lon_col, shop_lat_col = find_coord_columns(shop_data, "商店")
    bus_lon_col, bus_lat_col = find_coord_columns(bus_data, "公交站")
    school_lon_col, school_lat_col = find_coord_columns(school_data, "学校")
except ValueError as e:
    print(f"数据列名错误: {str(e)}")
    print("请确保CSV文件包含经纬度坐标列")
    raise

print(f"商店经度列: {shop_lon_col}, 纬度列: {shop_lat_col}")
print(f"公交站经度列: {bus_lon_col}, 纬度列: {bus_lat_col}")
print(f"学校经度列: {school_lon_col}, 纬度列: {school_lat_col}")

# 3. 数据清洗和验证
def clean_coordinate_data(df, lon_col, lat_col, data_type):
    """
    清洗坐标数据，移除无效值
    """
    print(f"清洗{data_type}坐标数据...")
    
    # 检查并转换坐标数据类型
    try:
        df[lon_col] = pd.to_numeric(df[lon_col], errors='coerce')
        df[lat_col] = pd.to_numeric(df[lat_col], errors='coerce')
    except Exception as e:
        print(f"转换{data_type}坐标数据类型时出错: {str(e)}")
        raise
    
    # 移除无效坐标（NaN, 无穷大等）
    original_count = len(df)
    df = df.dropna(subset=[lon_col, lat_col])
    
    # 检查坐标范围（纽约的经纬度范围：经度-74到-73，纬度40到41）
    df = df[(df[lon_col] >= -75) & (df[lon_col] <= -72) & 
            (df[lat_col] >= 40) & (df[lat_col] <= 42)]
    
    removed_count = original_count - len(df)
    print(f"{data_type}数据清洗完成: 原始{original_count}条，移除{removed_count}条无效数据，剩余{len(df)}条")
    
    return df

# 清洗数据
shop_data = clean_coordinate_data(shop_data, shop_lon_col, shop_lat_col, "商店")
bus_data = clean_coordinate_data(bus_data, bus_lon_col, bus_lat_col, "公交站")
school_data = clean_coordinate_data(school_data, school_lon_col, school_lat_col, "学校")

# 4. 创建GeoDataFrame
shop_geometry = [Point(xy) for xy in zip(shop_data[shop_lon_col], shop_data[shop_lat_col])]
shop_gdf = gpd.GeoDataFrame(shop_data, geometry=shop_geometry, crs='EPSG:4326')

bus_geometry = [Point(xy) for xy in zip(bus_data[bus_lon_col], bus_data[bus_lat_col])]
bus_gdf = gpd.GeoDataFrame(bus_data, geometry=bus_geometry, crs='EPSG:4326')

school_geometry = [Point(xy) for xy in zip(school_data[school_lon_col], school_data[school_lat_col])]
school_gdf = gpd.GeoDataFrame(school_data, geometry=school_geometry, crs='EPSG:4326')

print("数据加载和清洗完成")

# 5. 计算距离的通用函数
def haversine_distance(lon1, lat1, lon2, lat2):
    """
    使用Haversine公式计算两点间的距离（米）
    """
    # 转换为弧度
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    
    # Haversine公式
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    # 地球半径（米）
    r = 6371000
    
    return c * r

def calculate_accessibility(facility_gdf, school_gdf, facility_name):
    """
    计算学校到最近设施的距离
    使用KDTree进行快速最近邻搜索
    """
    print(f"开始计算学校到最近{facility_name}的距离...")
    
    if len(facility_gdf) == 0 or len(school_gdf) == 0:
        raise ValueError(f"{facility_name}或学校数据为空，无法计算可达性")
    
    # 提取坐标
    facility_coords = np.array([[point.x, point.y] for point in facility_gdf.geometry])
    school_coords = np.array([[point.x, point.y] for point in school_gdf.geometry])
    
    # 使用KDTree进行最近邻搜索
    facility_tree = cKDTree(facility_coords)
    
    # 计算每个学校到最近设施的距离
    distances_deg, indices = facility_tree.query(school_coords)
    
    # 使用Haversine公式计算精确距离
    distances_meters = []
    for i, idx in enumerate(indices):
        dist_m = haversine_distance(
            school_coords[i][0], school_coords[i][1],
            facility_coords[idx][0], facility_coords[idx][1]
        )
        distances_meters.append(dist_m)
    
    distances_meters = np.array(distances_meters)
    
    # 添加距离信息到学校GeoDataFrame
    distance_col_name = f'nearest_{facility_name.lower()}_distance_m'
    index_col_name = f'nearest_{facility_name.lower()}_index'
    
    school_gdf[distance_col_name] = distances_meters
    school_gdf[index_col_name] = indices
    
    # 获取最近设施的信息
    nearest_facility_names = facility_gdf.iloc[indices][facility_gdf.columns[0]].values
    school_gdf[f'nearest_{facility_name.lower()}'] = nearest_facility_names
    
    print(f"{facility_name}距离计算完成，最小距离: {distances_meters.min():.2f}米, 最大距离: {distances_meters.max():.2f}米")
    print(f"平均距离: {distances_meters.mean():.2f}米")
    
    return school_gdf, facility_gdf

# 计算学校到商店和公交站的可达性
try:
    school_gdf, shop_gdf = calculate_accessibility(shop_gdf, school_gdf, "Shop")
    school_gdf, bus_gdf = calculate_accessibility(bus_gdf, school_gdf, "Bus")
except ValueError as e:
    print(f"计算可达性时出错: {str(e)}")
    raise

# 6. 创建栅格化函数
def create_accessibility_grid(school_gdf, facility_gdf, cell_size=10, buffer_distance=5000):  # 修改默认值为10米
    """
    将学校可达性数据转换为栅格
    使用反距离加权插值方法
    """
    print("开始创建栅格数据...")
    
    if len(school_gdf) == 0 or len(facility_gdf) == 0:
        raise ValueError("学校或设施数据为空，无法创建栅格")
    
    # 获取研究区域边界
    school_bounds = school_gdf.total_bounds  # (minx, miny, maxx, maxy)
    facility_bounds = facility_gdf.total_bounds
    
    # 综合考虑学校和设施的边界
    minx = min(school_bounds[0], facility_bounds[0])
    miny = min(school_bounds[1], facility_bounds[1])
    maxx = max(school_bounds[2], facility_bounds[2])
    maxy = max(school_bounds[3], facility_bounds[3])
    
    # 扩展边界以包含缓冲区
    lat_factor = np.cos(np.radians((miny + maxy) / 2))
    minx -= buffer_distance / (111000 * lat_factor)
    miny -= buffer_distance / 111000
    maxx += buffer_distance / (111000 * lat_factor)
    maxy += buffer_distance / 111000
    
    # 计算栅格尺寸（使用Haversine距离转换）
    x_range = haversine_distance(minx, (miny + maxy) / 2, maxx, (miny + maxy) / 2)
    y_range = haversine_distance((minx + maxx) / 2, miny, (minx + maxx) / 2, maxy)
    
    width = max(10, int(x_range / cell_size))  # 确保至少有10个像素
    height = max(10, int(y_range / cell_size))
    
    print(f"研究区域范围: ({minx:.6f}, {miny:.6f}, {maxx:.6f}, {maxy:.6f})")
    print(f"栅格尺寸: {width} x {height}, 像素大小: {cell_size}米")
    
    # 创建坐标网格
    x_coords = np.linspace(minx, maxx, width)
    y_coords = np.linspace(maxy, miny, height)  # 注意：y轴从上到下
    xx, yy = np.meshgrid(x_coords, y_coords)
    
    # 展平坐标用于距离计算
    grid_points = np.column_stack([xx.ravel(), yy.ravel()])
    
    # 提取学校坐标和距离值
    school_coords = np.array([[point.x, point.y] for point in school_gdf.geometry])
    
    # 获取对应的距离列（查找包含distance_m的列）
    distance_cols = [col for col in school_gdf.columns if 'distance_m' in col]
    if len(distance_cols) < 2:
        raise ValueError("没有找到足够的距离列")
    
    # 分别处理商店和公交站的距离
    shop_distance_col = [col for col in distance_cols if 'shop' in col.lower()][0]
    bus_distance_col = [col for col in distance_cols if 'bus' in col.lower()][0]
    
    shop_distances = school_gdf[shop_distance_col].values
    bus_distances = school_gdf[bus_distance_col].values
    
    print(f"商店距离列: {shop_distance_col}")
    print(f"公交站距离列: {bus_distance_col}")
    
    # 为商店和公交站分别创建栅格
    accessibility_grids = {}
    
    for name, distances in [("shop", shop_distances), ("bus", bus_distances)]:
        print(f"正在创建{name}可达性栅格...")
        
        # 创建KDTree
        school_tree = cKDTree(school_coords)
        
        # 对于每个栅格点，计算插值
        accessibility_grid = np.full(len(grid_points), np.nan)
        
        # 设置插值参数
        k_neighbors = min(10, len(school_coords))  # 使用最近的10个点
        power = 2  # IDW的幂参数
        
        print(f"正在进行{name}空间插值...")
        print(f"总栅格点数: {len(grid_points)} (栅格尺寸: {height} x {width})")
        
        # 分批处理以节省内存
        batch_size = 2000  # 减小批次大小以适应10米高精度
        for start_idx in range(0, len(grid_points), batch_size):
            end_idx = min(start_idx + batch_size, len(grid_points))
            
            for i in range(start_idx, end_idx):
                if i % 1000 == 0:  # 每处理1000个点打印一次进度
                    print(f"已处理 {i}/{len(grid_points)} 个栅格点 ({name})")
                
                # 找到最近的k个学校点
                distances_to_schools, indices = school_tree.query(grid_points[i], k=k_neighbors)
                
                # 处理单个点的情况
                if np.isscalar(distances_to_schools):
                    distances_to_schools = np.array([distances_to_schools])
                    indices = np.array([indices])
                
                # 计算加权平均
                weights = []
                values = []
                
                for j, (dist, idx) in enumerate(zip(distances_to_schools, indices)):
                    if dist < 1e-6:  # 如果距离非常小，直接使用该点的值
                        accessibility_grid[i] = distances[idx]
                        break
                    else:
                        # 使用反距离加权
                        weight = 1 / (dist ** power)
                        weights.append(weight)
                        values.append(distances[idx])
                else:
                    # 如果没有距离为0的点，则计算加权平均
                    if len(weights) > 0:
                        total_weight = sum(weights)
                        if total_weight > 0:
                            accessibility_grid[i] = sum(w * v for w, v in zip(weights, values)) / total_weight
                        else:
                            accessibility_grid[i] = np.nan
                    else:
                        accessibility_grid[i] = np.nan
        
        # 重塑为2D栅格
        accessibility_grid = accessibility_grid.reshape(height, width)
        accessibility_grids[name] = accessibility_grid
    
    # 计算地理变换参数
    x_pixel_size = (maxx - minx) / width
    y_pixel_size = (maxy - miny) / height  # 注意这里是负值，因为y轴从上到下
    transform = from_origin(minx, maxy, x_pixel_size, abs(y_pixel_size))
    
    print("栅格数据创建完成")
    return accessibility_grids, transform, (minx, miny, maxx, maxy)

# 创建栅格数据 - 修改为10米精度
try:
    accessibility_grids, transform, bounds = create_accessibility_grid(school_gdf, shop_gdf, cell_size=10)  # 改为10米
except Exception as e:
    print(f"创建栅格时出错: {str(e)}")
    raise

# 7. 输出栅格数据
def save_raster_data(grid, transform, bounds, output_path):
    """
    保存栅格数据为GeoTIFF格式
    """
    print(f"开始保存栅格数据到: {output_path}")
    
    # 确保输出目录存在
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    with rasterio.open(
        output_path,
        'w',
        driver='GTiff',
        height=grid.shape[0],
        width=grid.shape[1],
        count=1,
        dtype=rasterio.float32,
        crs='EPSG:4326',
        transform=transform,
        nodata=-9999
    ) as dst:
        # 将NaN值替换为nodata值
        grid_masked = np.where(np.isnan(grid), -9999, grid).astype(np.float32)
        dst.write(grid_masked, 1)
    
    print("栅格数据保存完成")

# 保存可达性栅格
for facility_type, grid in accessibility_grids.items():
    output_raster_path = f'E:\\chuli\\accessibility\\NY_school_to_{facility_type}_accessibility_10m.tif'  # 修改文件名
    try:
        save_raster_data(grid, transform, bounds, output_raster_path)
        print(f"{facility_type}可达性栅格已保存: {output_raster_path}")
    except Exception as e:
        print(f"保存{facility_type}栅格数据时出错: {str(e)}")

# 8. 可视化结果
def visualize_results(school_gdf, shop_gdf, bus_gdf, accessibility_grids, bounds, save_path=None):
    """
    可视化学校到设施的可达性分析结果
    """
    print("开始可视化...")
    
    fig, axes = plt.subplots(2, 2, figsize=(20, 16))
    
    # 左上图：原始数据分布
    ax1 = axes[0, 0]
    ax1.scatter(shop_gdf.geometry.x, shop_gdf.geometry.y, c='green', s=1, alpha=0.6, label='商店')
    ax1.scatter(bus_gdf.geometry.x, bus_gdf.geometry.y, c='blue', s=1, alpha=0.6, label='公交站')
    ax1.scatter(school_gdf.geometry.x, school_gdf.geometry.y, c='red', s=20, alpha=0.8, label='学校')
    ax1.set_title('学校、商店与公交站分布')
    ax1.set_xlabel('经度')
    ax1.set_ylabel('纬度')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 右上图：学校到商店可达性
    ax2 = axes[0, 1]
    shop_grid = accessibility_grids['shop']
    valid_data = shop_grid[~np.isnan(shop_grid)]
    if len(valid_data) > 0:
        vmax = np.nanpercentile(shop_grid, 95)
        im1 = ax2.imshow(shop_grid, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], 
                        cmap='RdYlBu_r', origin='upper', vmin=0, vmax=vmax)
    else:
        im1 = ax2.imshow(shop_grid, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], 
                        cmap='RdYlBu_r', origin='upper')
    
    ax2.scatter(school_gdf.geometry.x, school_gdf.geometry.y, c='black', s=10, alpha=0.8, label='学校')
    ax2.set_title('学校到最近商店距离（米）')
    ax2.set_xlabel('经度')
    ax2.set_ylabel('纬度')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.colorbar(im1, ax=ax2, label='距离（米）', shrink=0.8)
    
    # 左下图：学校到公交站可达性
    ax3 = axes[1, 0]
    bus_grid = accessibility_grids['bus']
    valid_data = bus_grid[~np.isnan(bus_grid)]
    if len(valid_data) > 0:
        vmax = np.nanpercentile(bus_grid, 95)
        im2 = ax3.imshow(bus_grid, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], 
                        cmap='RdYlBu_r', origin='upper', vmin=0, vmax=vmax)
    else:
        im2 = ax3.imshow(bus_grid, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], 
                        cmap='RdYlBu_r', origin='upper')
    
    ax3.scatter(school_gdf.geometry.x, school_gdf.geometry.y, c='black', s=10, alpha=0.8, label='学校')
    ax3.set_title('学校到最近公交站距离（米）')
    ax3.set_xlabel('经度')
    ax3.set_ylabel('纬度')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    plt.colorbar(im2, ax=ax3, label='距离（米）', shrink=0.8)
    
    # 右下图：综合可达性（商店+公交站平均距离）
    ax4 = axes[1, 1]
    combined_grid = (accessibility_grids['shop'] + accessibility_grids['bus']) / 2
    valid_data = combined_grid[~np.isnan(combined_grid)]
    if len(valid_data) > 0:
        vmax = np.nanpercentile(combined_grid, 95)
        im3 = ax4.imshow(combined_grid, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], 
                        cmap='RdYlBu_r', origin='upper', vmin=0, vmax=vmax)
    else:
        im3 = ax4.imshow(combined_grid, extent=[bounds[0], bounds[2], bounds[1], bounds[3]], 
                        cmap='RdYlBu_r', origin='upper')
    
    ax4.scatter(school_gdf.geometry.x, school_gdf.geometry.y, c='black', s=10, alpha=0.8, label='学校')
    ax4.set_title('学校综合可达性（商店+公交站平均距离）（米）')
    ax4.set_xlabel('经度')
    ax4.set_ylabel('纬度')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.colorbar(im3, ax=ax4, label='距离（米）', shrink=0.8)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"可视化结果已保存到: {save_path}")
    
    plt.show()

# 可视化结果
viz_path = r'E:\chuli\accessibility\NY_school_accessibility_visualization_10m.png'  # 修改文件名
os.makedirs(os.path.dirname(viz_path), exist_ok=True)
try:
    visualize_results(school_gdf, shop_gdf, bus_gdf, accessibility_grids, bounds, viz_path)
except Exception as e:
    print(f"可视化时出错: {str(e)}")

# 9. 生成统计报告
def generate_statistics(school_gdf):
    """
    生成学校可达性统计报告
    """
    # 获取距离列
    distance_cols = [col for col in school_gdf.columns if 'distance_m' in col]
    if len(distance_cols) < 2:
        print("警告: 没有找到足够的距离列")
        return {}
    
    shop_distance_col = [col for col in distance_cols if 'shop' in col.lower()][0]
    bus_distance_col = [col for col in distance_cols if 'bus' in col.lower()][0]
    
    shop_distances = school_gdf[shop_distance_col].dropna()
    bus_distances = school_gdf[bus_distance_col].dropna()
    
    if len(shop_distances) == 0 or len(bus_distances) == 0:
        print("警告: 没有有效的距离数据")
        return {}
    
    stats = {
        '学校总数': len(school_gdf),
        '有商店可达的学校数': len(shop_distances),
        '有公交站可达的学校数': len(bus_distances),
        '学校到商店平均距离（米）': shop_distances.mean(),
        '学校到公交站平均距离（米）': bus_distances.mean(),
        '学校到商店中位数距离（米）': shop_distances.median(),
        '学校到公交站中位数距离（米）': bus_distances.median(),
        '学校到商店最短距离（米）': shop_distances.min(),
        '学校到公交站最短距离（米）': bus_distances.min(),
        '学校到商店最长距离（米）': shop_distances.max(),
        '学校到公交站最长距离（米）': bus_distances.max(),
        '学校到商店标准差（米）': shop_distances.std(),
        '学校到公交站标准差（米）': bus_distances.std(),
        '到商店距离<500米的学校数': (shop_distances < 500).sum(),
        '到公交站距离<500米的学校数': (bus_distances < 500).sum(),
        '到商店距离<1000米的学校数': (shop_distances < 1000).sum(),
        '到公交站距离<1000米的学校数': (bus_distances < 1000).sum(),
    }
    
    print("\n=== 学校可达性统计报告 ===")
    for key, value in stats.items():
        if isinstance(value, float):
            print(f"{key}: {value:.2f}")
        else:
            print(f"{key}: {value}")
    
    return stats

# 生成统计报告
stats = generate_statistics(school_gdf)

# 10. 保存处理后的学校数据
school_output_path = r'E:\chuli\accessibility\NY_school_accessibility_10m.csv'  # 修改文件名
try:
    school_gdf.to_csv(school_output_path, index=False, encoding='utf-8-sig')
    print(f"\n学校可达性数据已保存到: {school_output_path}")
except Exception as e:
    print(f"保存学校数据时出错: {str(e)}")

# 11. 创建不同距离阈值的二进制栅格
def create_binary_accessibility_grids(accessibility_grids, transform, bounds, thresholds=[500, 1000, 2000]):
    """
    创建不同距离阈值的二进制可达性栅格
    """
    print("创建不同距离阈值的二进制栅格...")
    
    for facility_type, grid in accessibility_grids.items():
        for threshold in thresholds:
            # 创建二进制栅格（可达=1，不可达=0），处理NaN值
            binary_grid = np.where(np.isnan(grid), 0, 
                                  np.where(grid <= threshold, 1, 0)).astype(np.int8)
            
            # 保存二进制栅格
            binary_path = f'E:\\chuli\\accessibility\\NY_school_to_{facility_type}_accessibility_10m_within_{threshold}m.tif'  # 修改文件名
            try:
                save_raster_data(binary_grid, transform, bounds, binary_path)
                print(f"{facility_type}距离阈值{threshold}米的二进制栅格已保存: {binary_path}")
            except Exception as e:
                print(f"保存二进制栅格 {binary_path} 时出错: {str(e)}")

# 创建二进制栅格
create_binary_accessibility_grids(accessibility_grids, transform, bounds, [500, 1000, 2000])

print(f"\n=== 分析完成 ===")
print("所有栅格文件、可视化图像和统计数据已保存到 E:\\chuli\\accessibility 文件夹中")
print("包含以下文件：")
for facility_type in accessibility_grids.keys():
    print(f"- 学校到{facility_type}可达性栅格 (10m精度)")
    print(f"- 学校到{facility_type}不同距离阈值的二进制栅格")

# 12. 显示栅格基本信息
print(f"\n=== 栅格数据信息 ===")
for facility_type, grid in accessibility_grids.items():
    print(f"{facility_type}栅格尺寸: {grid.shape}")
    valid_data = grid[~np.isnan(grid)]
    if len(valid_data) > 0:
        print(f"{facility_type}有效数据范围: {np.nanmin(grid):.2f} - {np.nanmax(grid):.2f} 米")
    print(f"{facility_type}NaN值数量: {np.count_nonzero(np.isnan(grid))}")
    print(f"{facility_type}有效值数量: {np.count_nonzero(~np.isnan(grid))}")

print("\n所有处理完成！")
print("纽约学校到商店和公交站的可达性分析已完成。")
print("栅格精度为10米×10米。")


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from rasterio.transform import from_origin
import os
from shapely.geometry import LineString, MultiLineString
import warnings

# 忽略部分警告
warnings.filterwarnings('ignore')

def calculate_road_density_raster(input_shp, output_dir, pixel_size=10.0):
    """
    计算道路密度并输出为GeoTIFF
    :param input_shp: 输入SHP文件路径
    :param output_dir: 输出文件夹路径
    :param pixel_size: 像素大小（米），默认10m
    """
    # 1. 读取数据
    print(f"正在读取文件: {input_shp} ...")
    try:
        gdf = gpd.read_file(input_shp)
    except Exception as e:
        print(f"读取失败: {e}")
        return

    # 检查是否存在reclass字段
    if 'reclass' not in gdf.columns:
        print(f"警告: {input_shp} 中不存在reclass字段")
        return

    # 2. 坐标系检查与投影
    # 如果是地理坐标系（经纬度），则自动投影到合适的UTM投影以获取米单位
    if gdf.crs.is_geographic:
        print("检测到地理坐标系，正在自动投影到UTM...")
        gdf = gdf.to_crs(gdf.estimate_utm_crs())
        print(f"已投影到: {gdf.crs.name}")
    else:
        print(f"当前坐标系: {gdf.crs.name} (假设单位为米)")

    # 3. 数据预处理：打散MultiLineString并细分线段
    print("正在处理几何数据（线段细分）...")
    # 将MultiLineString炸开为LineString
    gdf = gdf.explode(index_parts=False)
    
    # 使用shapely 2.0+的segmentize方法，将长线段切割为短线段（长度<像素的一半，即5m），提高统计精度
    # 如果geopandas版本较低可能不支持直接调用 .segmentize()，这里使用apply
    # max_segment_length 设置为 pixel_size / 2
    try:
        # 新版Geopandas/Shapely
        gdf['geometry'] = gdf.geometry.segmentize(max_segment_length=pixel_size/2)
    except AttributeError:
        # 兼容旧版本，若无segmentize需手动处理(略)，建议更新环境
        print("警告: 您的Geopandas版本较旧，无法使用segmentize优化，计算精度可能受影响。建议更新库。")
    
    # 4. 提取所有线段的坐标并计算长度
    print("正在计算线段长度与分布...")
    # 获取坐标表 (index, x, y)
    coords = gdf.get_coordinates()
    
    # 通过索引偏移构建线段：(x1, y1) -> (x2, y2)
    # 只有当索引(index)相同时，才是同一条线的连续点
    coords['next_x'] = coords['x'].shift(-1)
    coords['next_y'] = coords['y'].shift(-1)
    coords['next_index'] = coords.index.shift(-1)
    
    # 筛选有效线段（同一要素内的点）
    segments = coords[coords.index == coords['next_index']].copy()
    
    if len(segments) == 0:
        print("没有有效的线段数据，跳过此文件")
        return
    
    # 计算每小段的长度
    segments['length'] = np.sqrt((segments['next_x'] - segments['x'])**2 + (segments['next_y'] - segments['y'])**2)
    
    # 计算每小段的中点，用中点代表该线段落入的网格
    segments['mid_x'] = (segments['x'] + segments['next_x']) / 2
    segments['mid_y'] = (segments['y'] + segments['next_y']) / 2
    
    # 5. 定义栅格网格范围
    minx, miny, maxx, maxy = gdf.total_bounds
    # 稍微向外扩展一点边界
    minx, miny = minx - pixel_size, miny - pixel_size
    maxx, maxy = maxx + pixel_size, maxy + pixel_size
    
    width = int(np.ceil((maxx - minx) / pixel_size))
    height = int(np.ceil((maxy - miny) / pixel_size))
    
    print(f"输出栅格大小: {width} x {height} (像素)")

    # 6. 二维直方图统计（加权求和）
    # 统计落入每个网格的线段长度之和
    # bins定义网格边界
    x_bins = np.linspace(minx, minx + width * pixel_size, width + 1)
    # 注意：栅格数据通常y轴是倒序的（从上到下），但在histogram2d中我们先按坐标统计，由rasterio处理写入顺序
    y_bins = np.linspace(miny, miny + height * pixel_size, height + 1)

    # 核心统计步骤：result是每个格网内的长度总和
    density_grid, x_edges, y_edges = np.histogram2d(
        segments['mid_x'], 
        segments['mid_y'], 
        bins=[x_bins, y_bins], 
        weights=segments['length']
    )
    
    # 7. 计算密度
    # 密度 = 长度 / 面积
    # 面积 = pixel_size * pixel_size
    cell_area = pixel_size * pixel_size
    density_grid = density_grid / cell_area
    
    # 注意：histogram2d返回的矩阵通过imshow显示时通常需要转置，
    # 且rasterio写入时，原点(minx, maxy)对应矩阵的第一行。
    # histogram2d的y_bins是从小到大，所以生成的矩阵第0行对应miny(底部)。
    # 需要将矩阵沿Y轴翻转，以便第0行对应地图顶部(maxy)。
    density_grid = np.flipud(density_grid.T)

    # 8. 输出结果
    filename = os.path.basename(input_shp).replace('.shp', '_density.tif')
    out_path = os.path.join(output_dir, filename)
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # 定义仿射变换
    transform = from_origin(minx, maxy, pixel_size, pixel_size)
    
    print(f"正在保存结果到: {out_path}")
    with rasterio.open(
        out_path,
        'w',
        driver='GTiff',
        height=density_grid.shape[0],
        width=density_grid.shape[1],
        count=1,
        dtype=rasterio.float32,
        crs=gdf.crs,
        transform=transform,
        compress='lzw'
    ) as dst:
        dst.write(density_grid.astype(rasterio.float32), 1)
    
    print("处理完成！\n")

def convert_road_with_reclass_to_raster(input_shp, output_dir, pixel_size=10.0):
    """
    将道路shp文件转换为栅格，保留reclass字段的文本属性
    """
    print(f"正在读取文件: {input_shp} ...")
    try:
        gdf = gpd.read_file(input_shp)
    except Exception as e:
        print(f"读取失败: {e}")
        return

    # 检查是否存在reclass字段
    if 'reclass' not in gdf.columns:
        print(f"错误: {input_shp} 中不存在reclass字段")
        return

    # 坐标系检查与投影
    if gdf.crs.is_geographic:
        print("检测到地理坐标系，正在自动投影到UTM...")
        gdf = gdf.to_crs(gdf.estimate_utm_crs())
        print(f"已投影到: {gdf.crs.name}")
    else:
        print(f"当前坐标系: {gdf.crs.name} (假设单位为米)")

    # 获取边界框
    minx, miny, maxx, maxy = gdf.total_bounds
    
    # 确保范围有效，如果范围太小则扩展
    x_range = maxx - minx
    y_range = maxy - miny
    
    if x_range == 0:
        maxx = minx + pixel_size
        x_range = pixel_size
    if y_range == 0:
        maxy = miny + pixel_size
        y_range = pixel_size
    
    # 计算输出栅格的尺寸，确保至少为1
    width = max(1, int(np.ceil(x_range / pixel_size)))
    height = max(1, int(np.ceil(y_range / pixel_size)))
    
    print(f"数据范围: X({minx:.2f}, {maxx:.2f}), Y({miny:.2f}, {maxy:.2f})")
    print(f"计算的栅格尺寸: {width} x {height}")
    
    # 创建仿射变换矩阵
    transform = from_origin(minx, maxy, pixel_size, pixel_size)
    
    # 获取唯一的reclass值并创建数值映射
    unique_values = gdf['reclass'].astype(str).unique()
    value_to_num = {val: i+1 for i, val in enumerate(unique_values)}
    num_to_value = {i+1: val for val, i in value_to_num.items()}
    
    # 创建查找表
    lookup_df = pd.DataFrame(list(num_to_value.items()), columns=['Numeric_Value', 'Reclass_Attribute'])
    base_name = os.path.splitext(os.path.basename(input_shp))[0]
    lookup_path = os.path.join(output_dir, f"{base_name}_lookup.csv")
    lookup_df.to_csv(lookup_path, index=False)
    
    # 准备几何体和对应的数值
    shapes = []
    for idx, row in gdf.iterrows():
        if row['geometry'] is not None and not row['geometry'].is_empty:
            if hasattr(row['geometry'], 'geom_type') and row['geometry'].geom_type in ['LineString', 'MultiLineString']:
                numeric_val = value_to_num[str(row['reclass'])]
                shapes.append((row['geometry'], numeric_val))
    
    # 创建栅格数据
    if shapes:
        print(f"准备栅格化 {len(shapes)} 个几何体")
        raster_data = features.rasterize(
            shapes=shapes,
            out_shape=(height, width),
            transform=transform,
            fill=0,
            dtype=rasterio.uint16
        )
    else:
        print("没有有效的几何体进行栅格化")
        raster_data = np.zeros((height, width), dtype=rasterio.uint16)
    
    print(f"最终栅格数据形状: {raster_data.shape}")
    
    # 输出文件路径
    output_raster = os.path.join(output_dir, f"{base_name}_raster.tif")
    
    # 保存为栅格文件
    with rasterio.open(
        output_raster,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=rasterio.uint16,
        crs=gdf.crs,
        transform=transform,
        nodata=0
    ) as dst:
        dst.write(raster_data, 1)
        dst.update_tags(description='Road raster with reclass attribute (numeric values)')
        dst.update_tags(pixel_size=f'{pixel_size}m')
        dst.update_tags(lookup_file=os.path.basename(lookup_path))
    
    print(f"成功创建栅格: {output_raster}")

# --- 主程序 ---
if __name__ == "__main__":
    # 配置输入输出路径
    files = [
        r"E:\chuli\shpfile\NY_road_cut.shp",
        r"E:\chuli\shpfile\HK_road.shp"
    ]
    output_folder = r"E:\chuli\road_acc"
    
    pixel_size_meters = 10.0
    
    # 创建输出目录
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    for shp_file in files:
        if os.path.exists(shp_file):
            print(f"\n正在处理: {shp_file}")
            convert_road_with_reclass_to_raster(shp_file, output_folder, pixel_size=pixel_size_meters)
        else:
            print(f"错误: 找不到文件 {shp_file}")


In [ ]:
import os
import geopandas as gpd
import pandas as pd
from collections import Counter

def count_fclass_in_shp_files(folder_path):
    """
    统计文件夹内所有shp文件的fclass字段个数
    
    参数:
    folder_path: 包含shp文件的文件夹路径
    """
    
    # 获取文件夹内所有shp文件
    shp_files = []
    for file in os.listdir(folder_path):
        if file.lower().endswith('.shp'):
            shp_files.append(os.path.join(folder_path, file))
    
    if not shp_files:
        print(f"在 {folder_path} 中没有找到任何shp文件")
        return
    
    print(f"找到 {len(shp_files)} 个shp文件")
    
    # 存储统计结果
    all_counts = []
    
    # 遍历每个shp文件
    for shp_file in shp_files:
        try:
            print(f"正在处理: {os.path.basename(shp_file)}")
            
            # 读取shp文件
            gdf = gpd.read_file(shp_file)
            
            # 检查是否存在fclass字段
            if 'fclass' not in gdf.columns:
                print(f"  警告: {os.path.basename(shp_file)} 中不存在fclass字段")
                continue
            
            # 统计fclass字段的值
            fclass_counts = gdf['fclass'].value_counts()
            
            # 将结果添加到总列表中
            for fclass_value, count in fclass_counts.items():
                all_counts.append({
                    'FileName': os.path.basename(shp_file),
                    'FClass': fclass_value,
                    'Count': count
                })
            
            print(f"  完成，统计了 {len(fclass_counts)} 种fclass类型")
            
        except Exception as e:
            print(f"  处理 {os.path.basename(shp_file)} 时出错: {str(e)}")
            continue
    
    # 创建DataFrame
    df = pd.DataFrame(all_counts)
    
    if df.empty:
        print("没有找到任何包含fclass字段的有效数据")
        return
    
    # 透视表：行是fclass，列是文件名，值是计数
    pivot_df = df.pivot_table(
        index='FClass', 
        columns='FileName', 
        values='Count', 
        fill_value=0,
        aggfunc='sum'
    )
    
    # 添加总计列
    pivot_df['Total'] = pivot_df.sum(axis=1)
    
    # 重新排序，按总计降序排列
    pivot_df = pivot_df.sort_values('Total', ascending=False)
    
    # 保存到桌面
    desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
    output_file = os.path.join(desktop_path, "fclass_statistics.xlsx")
    
    # 保存为Excel文件
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        # 透视表
        pivot_df.to_excel(writer, sheet_name='Pivot_Statistics')
        
        # 详细统计表
        df_sorted = df.sort_values(['FileName', 'Count'], ascending=[True, False])
        df_sorted.to_excel(writer, sheet_name='Detailed_Statistics', index=False)
    
    print(f"\n统计结果已保存到: {output_file}")
    print(f"共统计了 {len(pivot_df)} 种不同的fclass值")
    print(f"共处理了 {len(shp_files)} 个shp文件")
    
    # 显示前几行结果
    print("\n前10个fclass值的统计结果:")
    print(pivot_df.head(10))
    
    return pivot_df, df

def main():
    # 定义文件夹路径
    folder_path = r"E:\chuli\land_chaos"
    
    # 检查文件夹是否存在
    if not os.path.exists(folder_path):
        print(f"错误: 文件夹 {folder_path} 不存在")
        return
    
    # 执行统计
    result = count_fclass_in_shp_files(folder_path)
    
    if result:
        pivot_df, detail_df = result
        print(f"\n统计完成！结果已保存到桌面的 fclass_statistics.xlsx 文件中")
    else:
        print("统计过程中出现错误")

if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import geopandas as gpd
import os
import glob

# 检查是否安装了必要的库
try:
    import geopandas as gpd
except ImportError:
    print("正在安装geopandas...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "geopandas"])
    import geopandas as gpd

# 设置文件夹路径
folder_path = r'E:\chuli\land_chaos'

# 获取所有HK前缀的shp文件
hk_shp_files = glob.glob(os.path.join(folder_path, 'HK*.shp'))
print(f"找到HK前缀shp文件数量: {len(hk_shp_files)}")
print("HK shp文件列表:")
for file in hk_shp_files:
    print(f"  {file}")

# 获取所有NY前缀的shp文件
ny_shp_files = glob.glob(os.path.join(folder_path, 'NY*.shp'))
print(f"\n找到NY前缀shp文件数量: {len(ny_shp_files)}")
print("NY shp文件列表:")
for file in ny_shp_files:
    print(f"  {file}")

# 合并HK前缀的shp文件
if hk_shp_files:
    hk_gdfs = []
    for file in hk_shp_files:
        try:
            # 读取shp文件
            gdf = gpd.read_file(file)
            hk_gdfs.append(gdf)
            print(f"成功读取HK shp文件: {file}")
            print(f"  - 行数: {len(gdf)}, 列数: {len(gdf.columns)}")
        except Exception as e:
            print(f"读取shp文件失败 {file}: {str(e)}")
    
    if hk_gdfs:
        # 合并所有HK的GeoDataFrames
        print("\n正在合并HK shp文件...")
        hk_combined = pd.concat(hk_gdfs, ignore_index=True)
        
        # 保存合并后的数据为CSV格式（移除几何列）
        hk_csv_path = os.path.join(folder_path, 'HK_combined.csv')
        # 如果需要保留几何信息，可以使用geometry.to_wkt()转换为文本
        if 'geometry' in hk_combined.columns:
            hk_for_csv = hk_combined.drop(columns=['geometry'])
        else:
            hk_for_csv = hk_combined
        hk_for_csv.to_csv(hk_csv_path, index=False, encoding='utf-8-sig')
        print(f"HK文件合并完成，保存至: {hk_csv_path}")
        print(f"合并后数据行数: {len(hk_for_csv)}, 列数: {len(hk_for_csv.columns)}")
        
        # 如果需要保留几何信息，也可以保存为带几何信息的CSV
        hk_csv_with_geom = os.path.join(folder_path, 'HK_combined_with_geometry.csv')
        if 'geometry' in hk_combined.columns:
            hk_with_geom = hk_combined.copy()
            hk_with_geom['geometry'] = hk_with_geom['geometry'].apply(lambda x: x.wkt if x is not None else None)
            hk_with_geom.to_csv(hk_csv_with_geom, index=False, encoding='utf-8-sig')
            print(f"带几何信息的HK文件也已保存至: {hk_csv_with_geom}")
    else:
        print("没有成功读取到任何HK shp文件")
else:
    print("没有找到HK前缀的shp文件")

# 合并NY前缀的shp文件
if ny_shp_files:
    ny_gdfs = []
    for file in ny_shp_files:
        try:
            # 读取shp文件
            gdf = gpd.read_file(file)
            ny_gdfs.append(gdf)
            print(f"成功读取NY shp文件: {file}")
            print(f"  - 行数: {len(gdf)}, 列数: {len(gdf.columns)}")
        except Exception as e:
            print(f"读取shp文件失败 {file}: {str(e)}")
    
    if ny_gdfs:
        # 合并所有NY的GeoDataFrames
        print("\n正在合并NY shp文件...")
        ny_combined = pd.concat(ny_gdfs, ignore_index=True)
        
        # 保存合并后的数据为CSV格式（移除几何列）
        ny_csv_path = os.path.join(folder_path, 'NY_combined.csv')
        # 如果需要保留几何信息，可以使用geometry.to_wkt()转换为文本
        if 'geometry' in ny_combined.columns:
            ny_for_csv = ny_combined.drop(columns=['geometry'])
        else:
            ny_for_csv = ny_combined
        ny_for_csv.to_csv(ny_csv_path, index=False, encoding='utf-8-sig')
        print(f"NY文件合并完成，保存至: {ny_csv_path}")
        print(f"合并后数据行数: {len(ny_for_csv)}, 列数: {len(ny_for_csv.columns)}")
        
        # 如果需要保留几何信息，也可以保存为带几何信息的CSV
        ny_csv_with_geom = os.path.join(folder_path, 'NY_combined_with_geometry.csv')
        if 'geometry' in ny_combined.columns:
            ny_with_geom = ny_combined.copy()
            ny_with_geom['geometry'] = ny_with_geom['geometry'].apply(lambda x: x.wkt if x is not None else None)
            ny_with_geom.to_csv(ny_csv_with_geom, index=False, encoding='utf-8-sig')
            print(f"带几何信息的NY文件也已保存至: {ny_csv_with_geom}")
    else:
        print("没有成功读取到任何NY shp文件")
else:
    print("没有找到NY前缀的shp文件")

print("\n所有shp文件合并操作完成！")
print(f"输出文件已保存在: {folder_path}")


In [ ]:
import pandas as pd

def classify_landuse(poi_type):
    """
    根据POI类型进行土地利用分类
    """
    # 商业服务类
    commercial = ['clothes', 'jeweller', 'florist', 'greengrocer', 'cycle shop', 
                 'computer shop', 'shoe shop', 'butcher', 'bookshop', 'stationer', 
                 'video shop', 'outdoor shop', 'mobile phone', 'beauty shop', 
                 'car dealers', 'restaurant', 'cafe', 'fast food', 'pub', 'beverages', 
                 'ice rink', 'hairdresser', 'chemist', 'optician', 'dentist', 'doctors', 
                 'clinic', 'veterinary', 'car wash', 'hotel', 'guesthouse', 'bank', 
                 'atm', 'post office', 'post box', 'convenience', 'general', 'seaside shop']
    
    # 公共服务与机构类
    public_service = ['town hall', 'library', 'museum', 'theatre', 'art centre', 
                     'community centre', 'embassy', 'prison', 'memorial', 'monument', 
                     'hospital', 'christian', 'jewish', 'buddhist', 'islamic', 
                     'christian cath', 'christian prot', 'christian lutheran', 
                     'christian methodist', 'christian anglican', 'christian protestant', 
                     'christian evangelical', 'christian orthodox', 'islamic shia', 
                     'sikh', 'grave yard', 'college', 'kindergarten', 'fire station', 
                     'artwork', 'wayside cross']
    
    # 交通与市政设施类
    transportation = ['bus stop', 'tram stop', 'railway station', 'taxi', 'ferry terminal', 
                     'parking', 'bicycle parking', 'rental', 'crossing', 'traffic sign', 
                     'street lane', 'roundabout', 'slipway', 'lock gate', 'pier', 'marina', 
                     'helipad', 'tower', 'recycling', 'water well', 'spring', 'fountain', 
                     'dam', 'bus station', 'railway halt', 'airport', 'service', 'comms tower']
    
    # 休闲与开放空间类
    recreation = ['tourist info', 'viewpoint', 'park', 'garden', 'playground', 'beach', 
                 'cliff', 'island', 'waterfall', 'camping site', 'sports centre', 'pitch', 
                 'stadium', 'swimming pool', 'cycling path', 'ruins', 'fort', 'lighthouse', 
                 'cave entrance', 'theme park', 'track', 'cycling club']
    
    # 其他/基础设施类
    other_infrastructure = ['tree', 'bench', 'telephone', 'utility', 'shelter', 'apartment', 
                          'chalet', 'naval base', 'farm']
    
    # 需要忽略的类型（区域描述）
    ignore_types = ['suburb', 'city', 'region']
    
    if pd.isna(poi_type) or str(poi_type).strip() == '':
        return '未知类型'
    
    poi_type = str(poi_type).lower().strip()
    
    if poi_type in ignore_types:
        return 'ignore_types'
    elif poi_type in commercial:
        return 'commercial'
    elif poi_type in public_service:
        return 'public_service'
    elif poi_type in transportation:
        return 'transportation'
    elif poi_type in recreation:
        return 'recreation'
    elif poi_type in other_infrastructure:
        return 'other_infrastructure'
    else:
        return 'ignore_types'

def process_landuse_data(file_path):
    """
    处理单个CSV文件，添加landclass字段
    """
    try:
        # 读取CSV文件
        df = pd.read_csv(file_path)
        print(f"成功读取文件: {file_path}")
        print(f"数据形状: {df.shape}")
        print(f"列名: {list(df.columns)}")
        
        # 检查是否包含fclass列
        if 'fclass' not in df.columns:
            print(f"警告: 文件 {file_path} 中未找到 'fclass' 列")
            # 尝试查找可能的POI类型列
            possible_columns = [col for col in df.columns if 'type' in col.lower() or 'class' in col.lower()]
            if possible_columns:
                print(f"可能的POI类型列: {possible_columns}")
            return None
        
        # 添加landclass字段
        df['landclass'] = df['fclass'].apply(classify_landuse)
        
        # 统计各类别数量
        class_counts = df['landclass'].value_counts()
        print("\n土地利用分类统计:")
        for class_type, count in class_counts.items():
            print(f"{class_type}: {count}个")
        
        # 保存处理后的文件
        output_path = file_path.replace('.csv', '_with_landclass.csv')
        df.to_csv(output_path, index=False)
        print(f"\n处理完成！结果已保存至: {output_path}")
        
        return df
        
    except Exception as e:
        print(f"处理文件 {file_path} 时出错: {str(e)}")
        return None

# 主程序
if __name__ == "__main__":
    # 文件路径列表
    file_paths = [
        r"E:\chuli\land_chaos\HK_combined_with_geometry.csv",
        r"E:\chuli\land_chaos\NY_combined_with_geometry.csv"
    ]
    
    # 处理所有文件
    all_results = {}
    for file_path in file_paths:
        print("=" * 60)
        print(f"正在处理文件: {file_path}")
        print("=" * 60)
        
        result = process_landuse_data(file_path)
        if result is not None:
            all_results[file_path] = result
            print("\n" + "=" * 60)
    
    # 汇总统计
    if all_results:
        print("\n" + "=" * 60)
        print("所有文件处理完成！汇总统计:")
        print("=" * 60)
        
        total_summary = pd.DataFrame()
        for file_path, df in all_results.items():
            file_name = file_path.split('\\')[-1]
            summary = df['landclass'].value_counts().reset_index()
            summary.columns = ['landclass', file_name]
            if total_summary.empty:
                total_summary = summary
            else:
                total_summary = pd.merge(total_summary, summary, on='landclass', how='outer')
        
        # 填充NaN值为0
        total_summary = total_summary.fillna(0)
        print(total_summary)
        
        # 保存汇总统计
        summary_path = r"E:\chuli\land_chaos\landuse_classification_summary.csv"
        total_summary.to_csv(summary_path, index=False)
        print(f"\n汇总统计已保存至: {summary_path}")
    else:
        print("没有成功处理任何文件，请检查文件路径和格式。")

In [ ]:
import pandas as pd
import os

# 设置文件路径
input_folder = r'E:\chuli\land_chaos'
output_folder = os.path.join(input_folder, 'result')

# 创建输出文件夹
os.makedirs(output_folder, exist_ok=True)

# 定义要保留的土地类别
keep_categories = ['commercial', 'public_service', 'recreation', 'transportation']

# 要删除的类别
remove_categories = ['other_infrastructure', 'ignore_types']

print(f"将要保留的土地类别: {keep_categories}")
print(f"将要删除的土地类别: {remove_categories}")

# 处理HK文件
hk_input_file = os.path.join(input_folder, 'HK_combined_with_geometry_with_landclass.csv')
if os.path.exists(hk_input_file):
    print(f"\n正在处理文件: {hk_input_file}")
    
    # 读取HK数据
    hk_df = pd.read_csv(hk_input_file)
    print(f"原始HK数据行数: {len(hk_df)}")
    print(f"原始HK数据列数: {len(hk_df.columns)}")
    
    # 检查是否存在landclass字段
    if 'landclass' not in hk_df.columns:
        print("错误: HK文件中不存在landclass字段")
    else:
        print(f"landclass字段的唯一值: {hk_df['landclass'].unique()}")
        
        # 删除指定类别的数据
        hk_filtered = hk_df[~hk_df['landclass'].isin(remove_categories)]
        
        print(f"过滤后HK数据行数: {len(hk_filtered)}")
        print(f"删除的数据行数: {len(hk_df) - len(hk_filtered)}")
        
        # 检查过滤后的类别
        remaining_categories = hk_filtered['landclass'].unique()
        print(f"过滤后剩余的landclass类别: {remaining_categories}")
        
        # 检查是否只包含需要的类别
        unexpected_categories = set(remaining_categories) - set(keep_categories)
        if unexpected_categories:
            print(f"警告: 存在不在预期列表中的类别: {unexpected_categories}")
        
        # 保存处理后的HK数据
        hk_output_file = os.path.join(output_folder, 'HK_filtered.csv')
        hk_filtered.to_csv(hk_output_file, index=False, encoding='utf-8-sig')
        print(f"HK数据处理完成，保存至: {hk_output_file}")
else:
    print(f"错误: 找不到文件 {hk_input_file}")

# 处理NY文件
ny_input_file = os.path.join(input_folder, 'NY_combined_with_geometry_with_landclass.csv')
if os.path.exists(ny_input_file):
    print(f"\n正在处理文件: {ny_input_file}")
    
    # 读取NY数据
    ny_df = pd.read_csv(ny_input_file)
    print(f"原始NY数据行数: {len(ny_df)}")
    print(f"原始NY数据列数: {len(ny_df.columns)}")
    
    # 检查是否存在landclass字段
    if 'landclass' not in ny_df.columns:
        print("错误: NY文件中不存在landclass字段")
    else:
        print(f"landclass字段的唯一值: {ny_df['landclass'].unique()}")
        
        # 删除指定类别的数据
        ny_filtered = ny_df[~ny_df['landclass'].isin(remove_categories)]
        
        print(f"过滤后NY数据行数: {len(ny_filtered)}")
        print(f"删除的数据行数: {len(ny_df) - len(ny_filtered)}")
        
        # 检查过滤后的类别
        remaining_categories = ny_filtered['landclass'].unique()
        print(f"过滤后剩余的landclass类别: {remaining_categories}")
        
        # 检查是否只包含需要的类别
        unexpected_categories = set(remaining_categories) - set(keep_categories)
        if unexpected_categories:
            print(f"警告: 存在不在预期列表中的类别: {unexpected_categories}")
        
        # 保存处理后的NY数据
        ny_output_file = os.path.join(output_folder, 'NY_filtered.csv')
        ny_filtered.to_csv(ny_output_file, index=False, encoding='utf-8-sig')
        print(f"NY数据处理完成，保存至: {ny_output_file}")
else:
    print(f"错误: 找不到文件 {ny_input_file}")

# 显示最终统计信息
print(f"\n处理完成！")
print(f"输出文件保存在: {output_folder}")

# 如果两个文件都处理成功，显示最终统计
if os.path.exists(hk_input_file) and 'landclass' in locals() and 'hk_filtered' in locals():
    if os.path.exists(ny_input_file) and 'ny_filtered' in locals():
        print(f"\n最终统计:")
        print(f"HK数据行数: {len(hk_filtered)}")
        print(f"NY数据行数: {len(ny_filtered)}")
        print(f"总数据行数: {len(hk_filtered) + len(ny_filtered)}")


In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from scipy.spatial.distance import cdist
from scipy.stats import entropy
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_bounds
import os
from shapely.geometry import Point, Polygon
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 设置文件路径
input_folder = r'E:\chuli\land_chaos'
output_folder = os.path.join(input_folder, 'result')

# 创建输出文件夹
os.makedirs(output_folder, exist_ok=True)

# 定义要处理的土地类别
land_categories = ['commercial', 'public_service', 'recreation', 'transportation']

print(f"处理的土地类别: {land_categories}")
print(f"像素大小: 10m x 10m")

# 读取处理过的CSV文件并转换为GeoDataFrame
def read_and_convert_to_gdf(csv_file, geometry_col='geometry'):
    """读取CSV文件并转换为GeoDataFrame"""
    df = pd.read_csv(csv_file)
    
    # 检查是否存在geometry列
    if geometry_col in df.columns:
        # 如果geometry列存在，将其转换为几何对象
        gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkt(df[geometry_col]))
    else:
        print(f"警告: {csv_file} 中不存在geometry列，无法进行栅格化")
        return None
    
    # 确保CRS设置正确（这里假设使用WGS84，根据实际情况调整）
    gdf.crs = 'EPSG:4326'  # WGS84
    return gdf

# 读取HK数据
hk_csv_file = os.path.join(output_folder, 'HK_filtered.csv')
print(f"\n正在处理HK数据: {hk_csv_file}")

hk_gdf = read_and_convert_to_gdf(hk_csv_file)
if hk_gdf is not None:
    print(f"HK数据行数: {len(hk_gdf)}")
    print(f"HK数据范围: {hk_gdf.total_bounds}")
    
    # 转换坐标系统以便使用米制单位
    hk_gdf = hk_gdf.to_crs('EPSG:3857')  # Web Mercator投影，单位为米
    print(f"HK数据转换后的范围: {hk_gdf.total_bounds}")
else:
    print("无法读取HK数据，跳过处理")

# 读取NY数据
ny_csv_file = os.path.join(output_folder, 'NY_filtered.csv')
print(f"\n正在处理NY数据: {ny_csv_file}")

ny_gdf = read_and_convert_to_gdf(ny_csv_file)
if ny_gdf is not None:
    print(f"NY数据行数: {len(ny_gdf)}")
    print(f"NY数据范围: {ny_gdf.total_bounds}")
    
    # 转换坐标系统以便使用米制单位
    ny_gdf = ny_gdf.to_crs('EPSG:3857')  # Web Mercator投影，单位为米
    print(f"NY数据转换后的范围: {ny_gdf.total_bounds}")
else:
    print("无法读取NY数据，跳过处理")

def create_raster(gdf, pixel_size=10):
    """将GeoDataFrame转换为栅格"""
    if gdf is None or len(gdf) == 0:
        return None, None
    
    # 获取边界范围
    minx, miny, maxx, maxy = gdf.total_bounds
    
    # 计算栅格尺寸
    width = int((maxx - minx) / pixel_size)
    height = int((maxy - miny) / pixel_size)
    
    print(f"栅格尺寸: {width} x {height}")
    
    # 创建变换矩阵
    transform = from_bounds(minx, miny, maxx, maxy, width, height)
    
    # 为每个类别创建二进制栅格
    category_rasters = {}
    for category in land_categories:
        # 筛选当前类别的数据
        category_gdf = gdf[gdf['landclass'] == category].copy()
        
        if len(category_gdf) > 0:
            # 创建二进制掩膜
            shapes = [(geom, 1) for geom in category_gdf.geometry]
            if shapes:
                try:
                    # 创建二进制栅格
                    binary_raster = rasterize(
                        shapes, 
                        out_shape=(height, width), 
                        transform=transform,
                        fill=0,
                        dtype=rasterio.uint8
                    )
                    category_rasters[category] = binary_raster
                except Exception as e:
                    print(f"创建{category}栅格时出错: {e}")
                    category_rasters[category] = np.zeros((height, width), dtype=np.uint8)
            else:
                category_rasters[category] = np.zeros((height, width), dtype=np.uint8)
        else:
            category_rasters[category] = np.zeros((height, width), dtype=np.uint8)
    
    return category_rasters, transform

def calculate_shannon_entropy(gdf, pixel_size=10):
    """计算香农熵"""
    if gdf is None or len(gdf) == 0:
        return None, None
    
    # 创建所有类别的组合栅格
    category_rasters, transform = create_raster(gdf, pixel_size)
    
    if category_rasters is None:
        return None, None
    
    # 创建综合栅格，每个像素包含类别信息
    height, width = list(category_rasters.values())[0].shape
    combined_raster = np.zeros((height, width), dtype=np.uint8)
    
    # 为每个类别分配一个数字ID
    category_to_id = {cat: idx+1 for idx, cat in enumerate(land_categories)}
    id_to_category = {idx+1: cat for idx, cat in enumerate(land_categories)}
    
    # 将每个类别的栅格合并到综合栅格中
    for category, raster in category_rasters.items():
        mask = raster == 1
        combined_raster[mask] = category_to_id[category]
    
    # 计算各类别的面积比例
    unique_values, counts = np.unique(combined_raster, return_counts=True)
    
    # 计算每个类别的比例（排除0值，即空白区域）
    proportions = []
    category_names = []
    
    for val, count in zip(unique_values, counts):
        if val != 0:  # 排除空白区域
            category_name = id_to_category.get(val, f'Unknown_{val}')
            proportion = count / (height * width)
            proportions.append(proportion)
            category_names.append(category_name)
    
    print(f"类别分布比例:")
    for name, prop in zip(category_names, proportions):
        print(f"  {name}: {prop:.4f}")
    
    # 计算香农熵
    if proportions:
        proportions = np.array(proportions)
        # 避免log(0)的情况
        proportions = proportions[proportions > 0]
        shannon_entropy = -np.sum(proportions * np.log(proportions))
    else:
        shannon_entropy = 0
    
    return shannon_entropy, combined_raster

# 处理HK数据
if hk_gdf is not None:
    print(f"\n正在为HK数据创建栅格...")
    hk_entropy, hk_combined_raster = calculate_shannon_entropy(hk_gdf, pixel_size=10)
    
    if hk_entropy is not None:
        print(f"HK数据香农熵: {hk_entropy:.4f}")
        
        # 保存栅格文件
        hk_raster_path = os.path.join(output_folder, 'HK_raster.tif')
        if hk_combined_raster is not None:
            with rasterio.open(
                hk_raster_path,
                'w',
                driver='GTiff',
                height=hk_combined_raster.shape[0],
                width=hk_combined_raster.shape[1],
                count=1,
                dtype=hk_combined_raster.dtype,
                crs=hk_gdf.crs,
                transform=from_bounds(*hk_gdf.to_crs('EPSG:4326').total_bounds, 
                                    hk_combined_raster.shape[1], hk_combined_raster.shape[0])
            ) as dst:
                dst.write(hk_combined_raster, 1)
            print(f"HK栅格文件已保存: {hk_raster_path}")
        
        # 保存香农熵结果
        hk_entropy_path = os.path.join(output_folder, 'HK_shannon_entropy.txt')
        with open(hk_entropy_path, 'w', encoding='utf-8') as f:
            f.write(f"HK数据香农熵: {hk_entropy:.6f}\n")
            f.write(f"数据行数: {len(hk_gdf)}\n")
            f.write(f"栅格尺寸: {hk_combined_raster.shape[1]} x {hk_combined_raster.shape[0]}\n")
            f.write(f"像素大小: 10m x 10m\n")
        print(f"HK香农熵结果已保存: {hk_entropy_path}")
    else:
        print("HK数据香农熵计算失败")
else:
    print("HK数据为空，无法计算香农熵")

# 处理NY数据
if ny_gdf is not None:
    print(f"\n正在为NY数据创建栅格...")
    ny_entropy, ny_combined_raster = calculate_shannon_entropy(ny_gdf, pixel_size=10)
    
    if ny_entropy is not None:
        print(f"NY数据香农熵: {ny_entropy:.4f}")
        
        # 保存栅格文件
        ny_raster_path = os.path.join(output_folder, 'NY_raster.tif')
        if ny_combined_raster is not None:
            with rasterio.open(
                ny_raster_path,
                'w',
                driver='GTiff',
                height=ny_combined_raster.shape[0],
                width=ny_combined_raster.shape[1],
                count=1,
                dtype=ny_combined_raster.dtype,
                crs=ny_gdf.crs,
                transform=from_bounds(*ny_gdf.to_crs('EPSG:4326').total_bounds, 
                                    ny_combined_raster.shape[1], ny_combined_raster.shape[0])
            ) as dst:
                dst.write(ny_combined_raster, 1)
            print(f"NY栅格文件已保存: {ny_raster_path}")
        
        # 保存香农熵结果
        ny_entropy_path = os.path.join(output_folder, 'NY_shannon_entropy.txt')
        with open(ny_entropy_path, 'w', encoding='utf-8') as f:
            f.write(f"NY数据香农熵: {ny_entropy:.6f}\n")
            f.write(f"数据行数: {len(ny_gdf)}\n")
            f.write(f"栅格尺寸: {ny_combined_raster.shape[1]} x {ny_combined_raster.shape[0]}\n")
            f.write(f"像素大小: 10m x 10m\n")
        print(f"NY香农熵结果已保存: {ny_entropy_path}")
    else:
        print("NY数据香农熵计算失败")
else:
    print("NY数据为空，无法计算香农熵")

# 总结
print(f"\n处理完成！")
print(f"输出文件保存在: {output_folder}")
print(f"文件列表:")
if hk_gdf is not None and hk_entropy is not None:
    print(f"  - HK_raster.tif")
    print(f"  - HK_shannon_entropy.txt")
if ny_gdf is not None and ny_entropy is not None:
    print(f"  - NY_raster.tif") 
    print(f"  - NY_shannon_entropy.txt")

print(f"\n最终结果:")
if hk_entropy is not None:
    print(f"HK香农熵: {hk_entropy:.4f}")
if ny_entropy is not None:
    print(f"NY香农熵: {ny_entropy:.4f}")


In [ ]:
import numpy as np
import rasterio
from rasterio.transform import from_origin
import os
import pandas as pd
from pathlib import Path

def get_geo_transform_from_csv(csv_file_path, data_shape):
    """
    从CSV文件中读取地理信息来生成地理变换参数
    """
    try:
        df = pd.read_csv(csv_file_path)
        
        # 尝试从CSV中获取地理信息
        # 常见的列名模式
        lon_cols = [col for col in df.columns if 'lon' in col.lower() or 'lng' in col.lower() or 'x' in col.lower()]
        lat_cols = [col for col in df.columns if 'lat' in col.lower() or 'y' in col.lower()]
        
        if lon_cols and lat_cols:
            # 如果有经纬度列，使用它们来计算变换参数
            min_lon = df[lon_cols[0]].min()
            max_lon = df[lon_cols[0]].max()
            min_lat = df[lat_cols[0]].min()
            max_lat = df[lat_cols[0]].max()
            
            # 计算像素大小
            width, height = data_shape[1], data_shape[0]  # 注意：numpy是(height, width)
            pixel_size_x = (max_lon - min_lon) / width
            pixel_size_y = (max_lat - min_lat) / height  # 通常为负值
            
            # 返回地理变换参数
            transform = from_origin(min_lon, max_lat, pixel_size_x, abs(pixel_size_y))
            crs = 'EPSG:4326'
            return transform, crs
        else:
            # 如果没有找到经纬度信息，使用默认参数
            print(f"在CSV文件 {csv_file_path} 中未找到经纬度信息，使用默认参数")
            pixel_size_x = 0.01
            pixel_size_y = 0.01
            top_left_x = 113.0
            top_left_y = 23.0
            transform = from_origin(top_left_x, top_left_y, pixel_size_x, pixel_size_y)
            crs = 'EPSG:4326'
            return transform, crs
            
    except Exception as e:
        print(f"读取CSV文件 {csv_file_path} 时出错: {str(e)}，使用默认参数")
        # 使用默认参数
        pixel_size_x = 0.01
        pixel_size_y = 0.01
        top_left_x = 113.0
        top_left_y = 23.0
        transform = from_origin(top_left_x, top_left_y, pixel_size_x, pixel_size_y)
        crs = 'EPSG:4326'
        return transform, crs

def npy_to_tif_with_csv(npy_file_path, output_folder):
    """
    将npy文件转换为tif格式，使用同名CSV文件的地理信息
    """
    # 读取npy文件
    data = np.load(npy_file_path)
    
    # 获取文件名（不含扩展名）
    file_name = Path(npy_file_path).stem
    output_tif_path = os.path.join(output_folder, f"{file_name}.tif")
    
    # 查找同名的CSV文件
    csv_file_path = os.path.join(output_folder, f"{file_name}.csv")
    
    # 获取地理变换参数
    if os.path.exists(csv_file_path):
        transform, crs = get_geo_transform_from_csv(csv_file_path, data.shape)
    else:
        print(f"未找到同名CSV文件: {csv_file_path}，使用默认参数")
        # 使用默认参数
        pixel_size_x = 0.01
        pixel_size_y = 0.01
        top_left_x = 113.0
        top_left_y = 23.0
        transform = from_origin(top_left_x, top_left_y, pixel_size_x, pixel_size_y)
        crs = 'EPSG:4326'
    
    # 获取数据的形状
    if len(data.shape) == 2:
        height, width = data.shape
        count = 1  # 单波段
        data_to_write = data
    elif len(data.shape) == 3:
        # 判断哪个维度是波段维度
        if data.shape[0] <= 4 and data.shape[0] < data.shape[1] and data.shape[0] < data.shape[2]:
            # 通道在第一个维度 (bands, height, width)
            count, height, width = data.shape
            data_to_write = data  # 不需要转置，rasterio期望(bands, height, width)
        elif data.shape[2] <= 4 and data.shape[2] < data.shape[0] and data.shape[2] < data.shape[1]:
            # 通道在最后一个维度 (height, width, bands)
            height, width, count = data.shape
            data_to_write = np.transpose(data, (2, 0, 1))  # 转换为(bands, height, width)
        else:
            # 假设是(height, width, channels)格式，但通道数较多
            height, width, count = data.shape
            data_to_write = np.transpose(data, (2, 0, 1))
    else:
        raise ValueError(f"不支持的数据形状: {data.shape}")
    
    # 获取数据类型
    dtype = data.dtype
    
    # 创建tif文件
    with rasterio.open(
        output_tif_path,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=count,
        dtype=dtype,
        crs=crs,
        transform=transform,
    ) as dst:
        if count == 1:
            # 单波段
            dst.write(data_to_write, 1)
        else:
            # 多波段
            dst.write(data_to_write)
    
    print(f"已转换: {npy_file_path} -> {output_tif_path}")

def batch_convert_npy_to_tif_with_csv(input_folder, output_folder):
    """
    批量转换npy文件为tif格式，使用同名CSV文件的地理信息
    """
    # 确保输出文件夹存在
    os.makedirs(output_folder, exist_ok=True)
    
    # 获取所有npy文件
    npy_files = [f for f in os.listdir(input_folder) if f.lower().endswith('.npy')]
    
    if not npy_files:
        print(f"在文件夹 {input_folder} 中未找到npy文件")
        return
    
    print(f"找到 {len(npy_files)} 个npy文件")
    
    for i, npy_file in enumerate(npy_files, 1):
        npy_file_path = os.path.join(input_folder, npy_file)
        print(f"[{i}/{len(npy_files)}] 正在转换: {npy_file}")
        
        try:
            npy_to_tif_with_csv(npy_file_path, output_folder)
        except Exception as e:
            print(f"转换失败 {npy_file}: {str(e)}")
            continue
    
    print("转换完成！")

# 主程序
input_folder = r"E:\chuli\traffic"
output_folder = r"E:\chuli\traffic"

# 执行批量转换
batch_convert_npy_to_tif_with_csv(input_folder, output_folder)


In [ ]:
import numpy as np
import rasterio
import os
import pandas as pd
from pathlib import Path

def analyze_tif_file(tif_path):
    """
    分析单个tif文件的统计信息
    """
    try:
        with rasterio.open(tif_path) as src:
            # 读取所有波段的数据
            data = src.read()  # shape: (bands, height, width)
            
            # 获取文件信息
            filename = Path(tif_path).name
            bands = src.count
            height = src.height
            width = src.width
            dtype = str(src.dtypes[0])
            
            # 计算统计信息
            stats = {
                'filename': filename,
                'bands': bands,
                'height': height,
                'width': width,
                'dtype': dtype
            }
            
            # 如果是多波段，对每个波段进行统计，如果是单波段则直接统计
            if bands == 1:
                band_data = data[0]  # 取第一个波段
                # 过滤掉无效值（如NaN或指定的nodata值）
                if src.nodata is not None:
                    band_data = band_data[band_data != src.nodata]
                
                if len(band_data.flatten()) > 0:
                    stats['band1_min'] = float(np.nanmin(band_data))
                    stats['band1_max'] = float(np.nanmax(band_data))
                    stats['band1_mean'] = float(np.nanmean(band_data))
                    stats['band1_std'] = float(np.nanstd(band_data))
                    stats['band1_median'] = float(np.nanmedian(band_data))
                else:
                    stats['band1_min'] = stats['band1_max'] = stats['band1_mean'] = \
                    stats['band1_std'] = stats['band1_median'] = np.nan
            else:
                # 多波段情况
                for band_idx in range(bands):
                    band_data = data[band_idx]
                    if src.nodata is not None:
                        band_data = band_data[band_data != src.nodata]
                    
                    if len(band_data.flatten()) > 0:
                        stats[f'band{band_idx+1}_min'] = float(np.nanmin(band_data))
                        stats[f'band{band_idx+1}_max'] = float(np.nanmax(band_data))
                        stats[f'band{band_idx+1}_mean'] = float(np.nanmean(band_data))
                        stats[f'band{band_idx+1}_std'] = float(np.nanstd(band_data))
                        stats[f'band{band_idx+1}_median'] = float(np.nanmedian(band_data))
                    else:
                        stats[f'band{band_idx+1}_min'] = stats[f'band{band_idx+1}_max'] = \
                        stats[f'band{band_idx+1}_mean'] = stats[f'band{band_idx+1}_std'] = \
                        stats[f'band{band_idx+1}_median'] = np.nan
            
            # 添加地理信息
            if src.crs:
                stats['crs'] = str(src.crs)
            else:
                stats['crs'] = 'Unknown'
            
            return stats
            
    except Exception as e:
        print(f"处理文件 {tif_path} 时出错: {str(e)}")
        return {
            'filename': Path(tif_path).name,
            'error': str(e)
        }

def analyze_all_tif_files(folder_path):
    """
    分析文件夹中所有tif文件的统计信息
    """
    # 获取所有tif文件
    tif_files = []
    for file in os.listdir(folder_path):
        if file.lower().endswith(('.tif', '.tiff')):
            tif_files.append(os.path.join(folder_path, file))
    
    if not tif_files:
        print(f"在文件夹 {folder_path} 中未找到tif文件")
        return pd.DataFrame()
    
    print(f"找到 {len(tif_files)} 个tif文件，开始分析...")
    
    # 分析每个文件
    all_stats = []
    for i, tif_file in enumerate(tif_files, 1):
        print(f"[{i}/{len(tif_files)}] 正在分析: {Path(tif_file).name}")
        stats = analyze_tif_file(tif_file)
        all_stats.append(stats)
    
    # 创建DataFrame
    df = pd.DataFrame(all_stats)
    
    # 如果有错误列，将其放在最后
    if 'error' in df.columns:
        cols = [col for col in df.columns if col != 'error'] + ['error']
        df = df[cols]
    
    return df

def save_analysis_results(df, output_path):
    """
    保存分析结果到Excel文件
    """
    try:
        df.to_excel(output_path, index=False)
        print(f"分析结果已保存到: {output_path}")
    except Exception as e:
        print(f"保存Excel文件时出错: {str(e)}")
        # 如果Excel保存失败，尝试保存为CSV
        csv_path = output_path.replace('.xlsx', '.csv')
        df.to_csv(csv_path, index=False)
        print(f"分析结果已保存到CSV格式: {csv_path}")

# 主程序
folder_path = r"D:\data\tif_file"
output_path = r"D:\data\tif_file_statistics.xlsx"

# 分析所有tif文件
df_stats = analyze_all_tif_files(folder_path)

if not df_stats.empty:
    # 显示统计结果
    print("\n统计结果预览:")
    print(df_stats.head())
    
    print("\n统计结果摘要:")
    print(df_stats.describe())
    
    # 保存结果
    save_analysis_results(df_stats, output_path)
    
    # 显示一些关键统计信息
    print(f"\n总共分析了 {len(df_stats)} 个tif文件")
    print("分析的文件列表:")
    for filename in df_stats['filename']:
        print(f"  - {filename}")
else:
    print("没有找到任何tif文件或所有文件分析都失败了")


In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from rasterio.mask import mask
import os
from shapely.geometry import box
import warnings
warnings.filterwarnings('ignore')

def analyze_road_raster_statistics(raster_path, lookup_csv_path, output_stats_path):
    """
    分析道路栅格的统计信息（基于reclass数值）
    """
    print(f"正在分析: {os.path.basename(raster_path)}")
    
    # 读取栅格数据
    with rasterio.open(raster_path) as src:
        reclass_data = src.read(1)  # 读取reclass数值
        transform = src.transform
        crs = src.crs
        nodata = src.nodata
    
    # 读取查找表
    lookup_df = pd.read_csv(lookup_csv_path)
    
    # 创建数值到reclass的映射
    num_to_reclass = dict(zip(lookup_df['Numeric_Value'], lookup_df['Reclass_Attribute']))
    
    # 获取所有唯一的reclass数值（排除nodata）
    if nodata is not None:
        unique_nums = np.unique(reclass_data[reclass_data != nodata])
    else:
        unique_nums = np.unique(reclass_data)
    
    unique_nums = unique_nums[unique_nums > 0]  # 排除0值（无数据区域）
    
    # 统计基本信息
    total_pixels = reclass_data.size
    valid_pixels = np.count_nonzero(reclass_data) if nodata is None else np.sum(reclass_data != nodata)
    total_area_km2 = (total_pixels * 100) / 1000000  # 10m像素 = 100m²
    valid_area_km2 = (valid_pixels * 100) / 1000000
    
    stats_summary = {
        '总像素数': total_pixels,
        '有效像素数': valid_pixels,
        '总面积_平方公里': total_area_km2,
        '有效面积_平方公里': valid_area_km2,
        '数据类型数量': len(unique_nums),
        '无数据值': nodata,
        '城市名称': os.path.basename(raster_path).replace('_raster.tif', '')
    }
    
    # 按reclass统计信息
    reclass_stats = []
    
    for num in unique_nums:
        if num in num_to_reclass:
            reclass_name = num_to_reclass[num]
            
            # 计算该reclass的像素数量和面积
            mask_reclass = (reclass_data == num)
            pixels_count = np.sum(mask_reclass)
            area_km2 = (pixels_count * 100) / 1000000
            
            reclass_stat = {
                '城市名称': os.path.basename(raster_path).replace('_raster.tif', ''),
                '道路等级': reclass_name,
                '数值编码': int(num),
                '像素数量': pixels_count,
                '面积_平方公里': area_km2,
                '占有效面积百分比': round((pixels_count / valid_pixels * 100) if valid_pixels > 0 else 0, 2),
                '密度_km_per_km2': round((pixels_count * 10) / area_km2 if area_km2 > 0 else 0, 2)  # 假设每像素代表10m道路长度
            }
            reclass_stats.append(reclass_stat)
    
    # 创建统计结果DataFrame
    reclass_df = pd.DataFrame(reclass_stats)
    
    # 保存统计结果到临时文件
    temp_output = output_stats_path.replace('.xlsx', '_temp.xlsx')
    with pd.ExcelWriter(temp_output, engine='openpyxl') as writer:
        # 总体统计
        summary_df = pd.DataFrame([stats_summary])
        summary_df.to_excel(writer, sheet_name='总体统计', index=False)
        
        # 按等级统计
        reclass_df.to_excel(writer, sheet_name='等级统计', index=False)
        
        # 详细分布
        value_counts = pd.Series(reclass_data.flatten()).value_counts()
        value_counts_df = pd.DataFrame({
            '数值编码': value_counts.index,
            '出现频次': value_counts.values
        })
        # 添加reclass名称
        value_counts_df['道路等级'] = value_counts_df['数值编码'].map(num_to_reclass)
        value_counts_df = value_counts_df[value_counts_df['数值编码'] > 0]  # 排除0值
        value_counts_df.to_excel(writer, sheet_name='数值分布', index=False)
    
    return stats_summary, reclass_df

def analyze_all_road_files(base_folder, output_desktop_folder):
    """
    分析所有道路文件的统计信息
    """
    # 查找所有栅格文件和对应的查找表
    raster_files = []
    for file in os.listdir(base_folder):
        if file.endswith('_raster.tif'):
            raster_path = os.path.join(base_folder, file)
            # 查找对应的查找表
            base_name = file.replace('_raster.tif', '')
            lookup_path = os.path.join(base_folder, f"{base_name}_lookup.csv")
            
            if os.path.exists(lookup_path):
                raster_files.append((raster_path, lookup_path))
    
    print(f"找到 {len(raster_files)} 个文件对进行分析")
    
    all_summary_results = []
    all_reclass_results = []
    
    for raster_path, lookup_path in raster_files:
        try:
            # 生成输出文件名
            base_name = os.path.basename(raster_path).replace('_raster.tif', '')
            output_stats_path = os.path.join(output_desktop_folder, f"{base_name}_statistics.xlsx")
            
            # 分析统计
            summary, reclass_df = analyze_road_raster_statistics(
                raster_path, lookup_path, output_stats_path
            )
            
            all_summary_results.append(summary)
            all_reclass_results.append(reclass_df)
            
            print(f"✓ 已完成分析: {base_name}")
            
        except Exception as e:
            print(f"✗ 分析失败 {os.path.basename(raster_path)}: {str(e)}")
    
    # 生成汇总统计文件
    if all_summary_results:
        summary_output = os.path.join(output_desktop_folder, "道路栅格统计汇总.xlsx")
        
        with pd.ExcelWriter(summary_output, engine='openpyxl') as writer:
            # 1. 总体统计: 所有城市的总体对比
            overall_summary_df = pd.DataFrame(all_summary_results)
            overall_summary_df.to_excel(writer, sheet_name='总体统计', index=False)
            
            # 2. 等级统计: 合并所有城市的等级统计
            if all_reclass_results:
                all_reclass_df = pd.concat(all_reclass_results, ignore_index=True)
                all_reclass_df.to_excel(writer, sheet_name='等级统计', index=False)
                
                # 3. 数值分布: 所有城市的数值分布汇总
                all_value_counts = []
                for raster_path, lookup_path in raster_files:
                    with rasterio.open(raster_path) as src:
                        reclass_data = src.read(1)
                        value_counts = pd.Series(reclass_data.flatten()).value_counts()
                        city_name = os.path.basename(raster_path).replace('_raster.tif', '')
                        for num, count in value_counts.items():
                            if num > 0:  # 排除0值
                                lookup_df = pd.read_csv(lookup_path)
                                num_to_reclass = dict(zip(lookup_df['Numeric_Value'], lookup_df['Reclass_Attribute']))
                                reclass_name = num_to_reclass.get(num, f'未知_{num}')
                                all_value_counts.append({
                                    '城市名称': city_name,
                                    '数值编码': num,
                                    '道路等级': reclass_name,
                                    '出现频次': count
                                })
                
                if all_value_counts:
                    all_value_counts_df = pd.DataFrame(all_value_counts)
                    all_value_counts_df.to_excel(writer, sheet_name='数值分布', index=False)
                
                # 4. 汇总统计: 所有城市的对比汇总
                summary_comparison = overall_summary_df.copy()
                summary_comparison.to_excel(writer, sheet_name='汇总统计', index=False)
                
                # 5. 等级汇总: 相同道路等级在不同城市的对比
                if len(all_reclass_df) > 0:
                    reclass_comparison = all_reclass_df.groupby('道路等级').agg({
                        '城市名称': lambda x: ', '.join(x.unique()),
                        '像素数量': 'sum',
                        '面积_平方公里': 'sum',
                        '占有效面积百分比': 'mean',
                        '密度_km_per_km2': 'mean'
                    }).reset_index()
                    reclass_comparison.columns = [
                        '道路等级', '涉及城市', '总像素数量', '总面积_平方公里', '平均占比', '平均密度'
                    ]
                    reclass_comparison.to_excel(writer, sheet_name='等级汇总', index=False)
        
        print(f"\n汇总统计文件已保存到: {summary_output}")

# 主程序
if __name__ == "__main__":
    # 设置输入输出路径
    input_folder = r"E:\\chuli\\road_acc"
    
    # 获取桌面路径
    desktop_path = os.path.join(os.path.expanduser("~"), "Desktop")
    output_desktop_folder = os.path.join(desktop_path, "道路统计结果")
    
    # 创建输出文件夹
    if not os.path.exists(output_desktop_folder):
        os.makedirs(output_desktop_folder)
    
    print("开始分析道路栅格统计信息...")
    print(f"输入文件夹: {input_folder}")
    print(f"输出文件夹: {output_desktop_folder}")
    
    # 分析所有文件
    analyze_all_road_files(input_folder, output_desktop_folder)
    
    print("\n分析完成！请查看桌面的 道路统计结果 文件夹。")
    print("\n汇总文件包含以下工作表:")
    print("1. 总体统计: 所有城市的总体对比统计")
    print("2. 等级统计: 每个城市每个道路等级的详细统计")
    print("3. 数值分布: 每个数值编码在各城市的出现频次")
    print("4. 汇总统计: 所有城市的对比汇总")
    print("5. 等级汇总: 相同道路等级在不同城市的对比")
